# Hannah — her short answers + visitors' names (free, can take several days)

Makes **898** talking clips: her **102 new short sentences** first, then **796 visitors' names** ("Divya!",
said the Indian way, sample B). About 3 hours of GPU in total: more than free Colab gives in one day,
so everything is **saved in your Google Drive** (folder *hannah_clips*). If Colab stops (GPU limit,
disconnect), just run it again later: it carries on where it stopped.

**How to run:**
1. **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**
2. A box asks to **connect Google Drive** → **Connect to Google Drive** → choose your account → **Continue / Allow**
3. First time only: at step 3 **Choose files** → **hannah-plate.png** (*Desktop → Hannah video test*)
4. Leave the tab open. Step 7 prints progress like `120/898 n0029 ok`.
5. Step 8 downloads what is ready: **hannah_short_clips.zip** (her sentences) and **hannah_names.zip**.
   Tell Claude when they are in Downloads. You can run step 8 any time to get what's done so far.


In [ ]:
# 1. Check the GPU
import subprocess, os
os.environ['MPLBACKEND'] = 'Agg'
try:
    gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu = ''
print(gpu or 'NO GPU! Videos need the GPU. Runtime → Change runtime type → T4 GPU. If Colab says the GPU limit is reached, try again tomorrow.')


In [ ]:
# 2. Your Google Drive keeps the work safe between sessions
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/hannah_clips'
for d in ['wav', 'web', 'web/names']:
    os.makedirs(W + '/' + d, exist_ok=True)
print('Saving to Google Drive → hannah_clips')


In [ ]:
# 2b. Install everything in its own clean Python 3.10 (5–8 minutes)
!apt-get -qq -y install espeak-ng ffmpeg > /dev/null 2>&1
!rm -rf /content/ditto && git clone -q --depth 1 https://github.com/antgroup/ditto-talkinghead /content/ditto
!pip install -q uv
!uv venv -q --seed --python 3.10 /content/venv
!uv pip install -q --python /content/venv/bin/python torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /content/venv/bin/python librosa filetype imageio imageio-ffmpeg scikit-image scipy cython setuptools einops tqdm opencv-python-headless mediapipe onnxruntime-gpu "kokoro>=0.9.4" "transformers<4.50" soundfile huggingface_hub
!/content/venv/bin/python -m spacy download en_core_web_sm -q > /dev/null 2>&1
!/content/venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('digital-avatar/ditto-talkinghead', local_dir='/content/ditto/checkpoints', allow_patterns=['ditto_pytorch/*', 'ditto_cfg/*'])"
!/content/venv/bin/python -c "import torch, filetype, librosa, cv2, mediapipe, onnxruntime, skimage, kokoro, soundfile, einops, pyximport; print('GPU for torch:', torch.cuda.is_available()); print('INSTALL OK')"

In [ ]:
# 3. Hannah's picture (asked only the first time; then it is kept in your Drive)
import shutil
from PIL import Image
if not os.path.exists(W + '/hannah.jpg'):
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    Image.open(name).convert('RGB').save(W + '/hannah.jpg', quality=95)
shutil.copy(W + '/hannah.jpg', '/content/hannah.jpg')
print('Picture ready', Image.open('/content/hannah.jpg').size)


In [ ]:
# 4. What she will say: her new sentences, then the names
import json
JOBS = json.loads(r'''[{"id": "c083", "say": "Hello, and welcome to Nexora! I'm Hannah, your virtual receptionist. May I know your name?", "lang": "a", "file": "c083.mp4"}, {"id": "c084", "say": "Welcome back! It's lovely to see you again. How can I help you today?", "lang": "a", "file": "c084.mp4"}, {"id": "c085", "say": "Nexora is real estate software, powered by Jatta M Kommerce. Would you like to hear more?", "lang": "a", "file": "c085.mp4"}, {"id": "c086", "say": "It helps developers show their projects in 3D, before they're even built. Would you like to hear more?", "lang": "a", "file": "c086.mp4"}, {"id": "c087", "say": "It also tracks construction and hands homes over to buyers, all in one place. Would you like to know what a 3D digital twin is?", "lang": "a", "file": "c087.mp4"}, {"id": "c088", "say": "A digital twin is a photorealistic 3D copy of a project. Would you like to hear more?", "lang": "a", "file": "c088.mp4"}, {"id": "c089", "say": "It's made from the architect's drawings, so buyers can explore every villa in the browser. Would you like to hear more?", "lang": "a", "file": "c089.mp4"}, {"id": "c090", "say": "Nexora can turn Revit, ArchiCAD or BIM models into a twin in under 72 hours. Would you like to see the virtual tour of our example villa?", "lang": "a", "file": "c090.mp4"}, {"id": "c091", "say": "Nexora has five main modules, from the empty plot to residents moving in. Would you like to hear more?", "lang": "a", "file": "c091.mp4"}, {"id": "c092", "say": "The first three are land and feasibility, a 3D sales gallery, and a construction tracker. Would you like to hear more?", "lang": "a", "file": "c092.mp4"}, {"id": "c093", "say": "Then come quality and snagging, and finally a buyer handover and resident portal. Shall I walk you through them one at a time?", "lang": "a", "file": "c093.mp4"}, {"id": "c094", "say": "Nexora is built for real estate developers and builders. Would you like to hear more?", "lang": "a", "file": "c094.mp4"}, {"id": "c095", "say": "From boutique villa builders to large township developers. Would you like to hear more?", "lang": "a", "file": "c095.mp4"}, {"id": "c096", "say": "Their buyers and residents use it too, to explore homes and follow construction. Are you a developer, or are you looking for a home yourself?", "lang": "a", "file": "c096.mp4"}, {"id": "c097", "say": "Phase 1 is Site Evaluation, where a developer checks the plot before building. Would you like to hear more?", "lang": "a", "file": "c097.mp4"}, {"id": "c098", "say": "Nexora tracks road access, surveys, water, power, approvals, zoning and soil tests. Would you like me to open Site Evaluation for you?", "lang": "a", "file": "c098.mp4"}, {"id": "c099", "say": "Phase 2 is Master Planning, where the villa layouts are finalised. Would you like to hear more?", "lang": "a", "file": "c099.mp4"}, {"id": "c100", "say": "RERA registration is secured, and the 3D twin goes live for buyers, with live inventory and booking. Shall I open Master Planning so you can see it?", "lang": "a", "file": "c100.mp4"}, {"id": "c101", "say": "Phase 3 is Construction, where the towers and clubhouse rise from the ground. Would you like to hear more?", "lang": "a", "file": "c101.mp4"}, {"id": "c102", "say": "Nexora tracks every milestone with drone images, bill checks and photo records. Would you like me to take you to the Construction phase?", "lang": "a", "file": "c102.mp4"}, {"id": "c103", "say": "Phase 4 is Fit-Out and Handover, when the interiors are finished. Would you like to hear more?", "lang": "a", "file": "c103.mp4"}, {"id": "c104", "say": "Every home is checked against more than 200 quality points, and the keys are handed over after a walkthrough. Shall I open Fit-Out and Handover for you?", "lang": "a", "file": "c104.mp4"}, {"id": "c105", "say": "Phase 5 is Ready to Move In, when residents settle in. Would you like to hear more?", "lang": "a", "file": "c105.mp4"}, {"id": "c106", "say": "The resident app handles amenity bookings, visitors, security and maintenance. Would you like to see Ready to Move In?", "lang": "a", "file": "c106.mp4"}, {"id": "c107", "say": "The Virtual Tour lets you explore our example luxury villa in 3D. Would you like to hear more?", "lang": "a", "file": "c107.mp4"}, {"id": "c108", "say": "You can walk through the garden, the clubhouse and pool, the living pavilion and the master suite. Shall I start the virtual tour for you?", "lang": "a", "file": "c108.mp4"}, {"id": "c109", "say": "The website tells Nexora's story in five phases, plus a virtual tour. Would you like to hear more?", "lang": "a", "file": "c109.mp4"}, {"id": "c110", "say": "Site Evaluation, Master Planning, Construction, Fit-Out and Handover, and Ready to Move In. Would you like to start at the beginning, with Site Evaluation?", "lang": "a", "file": "c110.mp4"}, {"id": "c111", "say": "Nexora has subscription plans for builders of every size. Would you like to hear more?", "lang": "a", "file": "c111.mp4"}, {"id": "c112", "say": "Prices aren't published, so the team shares pricing for your project on a demo. Would you like me to open the demo booking form?", "lang": "a", "file": "c112.mp4"}, {"id": "c113", "say": "I'm sorry, the villa here is an example, not a home for sale. Would you like to hear more?", "lang": "a", "file": "c113.mp4"}, {"id": "c114", "say": "So there are no prices or locations to share, but the virtual tour shows how buyers explore a project. Would you like to see the virtual tour?", "lang": "a", "file": "c114.mp4"}, {"id": "c115", "say": "I'd be happy to help you book a free demo. Would you like to hear more?", "lang": "a", "file": "c115.mp4"}, {"id": "c116", "say": "You share a little about your project, and the team contacts you for a personal walkthrough. Shall I open the demo form for you now?", "lang": "a", "file": "c116.mp4"}, {"id": "c117", "say": "You can email the team at contact at jattamkommerce dot com. Would you like to hear more?", "lang": "a", "file": "c117.mp4"}, {"id": "c118", "say": "Or use the Book a Demo form, and they'll get back to you. Would you like me to open the Book a Demo form?", "lang": "a", "file": "c118.mp4"}, {"id": "c119", "say": "Nexora is powered by Jatta M Kommerce. Would you like to hear more?", "lang": "a", "file": "c119.mp4"}, {"id": "c120", "say": "The team can tell you more at contact at jattamkommerce dot com. Would you like to know what Nexora does?", "lang": "a", "file": "c120.mp4"}, {"id": "c121", "say": "Nexora connects with tools like Salesforce, HubSpot and builder ERPs. Would you like to hear more?", "lang": "a", "file": "c121.mp4"}, {"id": "c122", "say": "There are also webhooks, REST APIs, and connectors for SAP and RERA portals. Shall I open the demo form, so the team can confirm what fits your setup?", "lang": "a", "file": "c122.mp4"}, {"id": "c123", "say": "Sustainability runs all through the example project. Would you like to hear more?", "lang": "a", "file": "c123.mp4"}, {"id": "c124", "say": "Rooftop solar, rainwater harvesting, greywater recycling, EV charging and green-building tracking. Would you like to start with Site Evaluation, where this planning begins?", "lang": "a", "file": "c124.mp4"}, {"id": "c125", "say": "Nexora accounts are still being set up, so log in isn't connected yet. Shall I open the demo form, so the team can give you access?", "lang": "a", "file": "c125.mp4"}, {"id": "c126", "say": "It's simple: just scroll to move through the phases. Would you like to hear more?", "lang": "a", "file": "c126.mp4"}, {"id": "c127", "say": "The Menu at the top right lists every phase, and Book a Demo is at the top of the page. Shall I take you into the website now?", "lang": "a", "file": "c127.mp4"}, {"id": "c128", "say": "This website doesn't save our conversation. Would you like to hear more?", "lang": "a", "file": "c128.mp4"}, {"id": "c129", "say": "When you speak, your browser turns your voice into text, so please don't share sensitive details here. Is there anything about Nexora I can help with?", "lang": "a", "file": "c129.mp4"}, {"id": "c130", "say": "Lovely, then Nexora is made for you. Would you like a quick look at the five modules, or shall I help you book a demo?", "lang": "a", "file": "c130.mp4"}, {"id": "c131", "say": "Wonderful! Nexora lets you explore homes in 3D, before they're built. Shall I start the virtual tour for you?", "lang": "a", "file": "c131.mp4"}, {"id": "c132", "say": "That's perfectly fine, take your time. Would you like me to show you around, starting with the first phase?", "lang": "a", "file": "c132.mp4"}, {"id": "c133", "say": "Hi there! Are you a developer or builder, looking for a home, or just exploring?", "lang": "a", "file": "c133.mp4"}, {"id": "c134", "say": "Hello! Great to have you here. Are you a developer, a home buyer, or just looking around?", "lang": "a", "file": "c134.mp4"}, {"id": "c135", "say": "I'm doing very well, thank you! How can I help you today?", "lang": "a", "file": "c135.mp4"}, {"id": "c136", "say": "I'm great, thanks for asking! What would you like to know about Nexora?", "lang": "a", "file": "c136.mp4"}, {"id": "c137", "say": "I'm Hannah, Nexora's virtual receptionist, and I'm here to guide you. What would you like to know?", "lang": "a", "file": "c137.mp4"}, {"id": "c138", "say": "I'm here to welcome you, answer your questions, and guide you around Nexora. What would you like to know?", "lang": "a", "file": "c138.mp4"}, {"id": "c139", "say": "I'm an AI receptionist, but I'm here to help just the same. What would you like to know?", "lang": "a", "file": "c139.mp4"}, {"id": "c140", "say": "Thank you for visiting Nexora! I'm always in the menu if you need me. Have a lovely day!", "lang": "a", "file": "c140.mp4"}, {"id": "c141", "say": "I can explain Nexora, guide you around the website, and help you book a demo. Where shall we start?", "lang": "a", "file": "c141.mp4"}, {"id": "c142", "say": "That's a little outside what I can help with, but I'd love to show you Nexora. Shall I give you a quick tour?", "lang": "a", "file": "c142.mp4"}, {"id": "c143", "say": "I'm sorry, I can't help with that one. But I can show you around Nexora. Shall we take a quick tour?", "lang": "a", "file": "c143.mp4"}, {"id": "c144", "say": "Ha, I wish I could help with that! I'm best at showing you Nexora. Would you like a quick tour?", "lang": "a", "file": "c144.mp4"}, {"id": "c145", "say": "No problem, let me keep it simple. Nexora helps builders show homes in 3D, before they're built. Shall I show you around?", "lang": "a", "file": "c145.mp4"}, {"id": "c146", "say": "That's completely fine. The easiest way to understand Nexora is a quick tour. Shall I show you around?", "lang": "a", "file": "c146.mp4"}, {"id": "c147", "say": "I'm sorry, I don't have that detail here. Our team can answer it on a demo. Shall I open the demo form?", "lang": "a", "file": "c147.mp4"}, {"id": "c148", "say": "Good question! I don't have the answer here, but the team will on a demo. Shall I open the demo form?", "lang": "a", "file": "c148.mp4"}, {"id": "c149", "say": "Sorry, I didn't quite catch that. Could you say it another way?", "lang": "a", "file": "c149.mp4"}, {"id": "c150", "say": "Sorry, I didn't follow. You can ask me what Nexora is, or ask me for a tour.", "lang": "a", "file": "c150.mp4"}, {"id": "c151", "say": "Great! I can explain Nexora, show you around, or help you book a demo. Which would you like?", "lang": "a", "file": "c151.mp4"}, {"id": "c152", "say": "Sorry, I can't understand your language. Please choose one of my languages, so we can talk.", "lang": "a", "file": "c152.mp4"}, {"id": "c153", "say": "Of course, taking you to Site Evaluation now.", "lang": "a", "file": "c153.mp4"}, {"id": "c154", "say": "Of course, taking you to Master Planning now.", "lang": "a", "file": "c154.mp4"}, {"id": "c155", "say": "Of course, taking you to Construction now.", "lang": "a", "file": "c155.mp4"}, {"id": "c156", "say": "Of course, taking you to Fit-Out and Handover now.", "lang": "a", "file": "c156.mp4"}, {"id": "c157", "say": "Of course, taking you to Ready to Move In now.", "lang": "a", "file": "c157.mp4"}, {"id": "c158", "say": "Of course, taking you to the Virtual Tour now.", "lang": "a", "file": "c158.mp4"}, {"id": "c159", "say": "Of course, opening the demo form for you.", "lang": "a", "file": "c159.mp4"}, {"id": "c160", "say": "Of course, taking you into the website now. I'm in the menu if you need me.", "lang": "a", "file": "c160.mp4"}, {"id": "c161", "say": "Sorry, I didn't catch your name. Could you type just your first name?", "lang": "a", "file": "c161.mp4"}, {"id": "c162", "say": "No problem at all! Are you a developer or builder, looking for a home, or just exploring?", "lang": "a", "file": "c162.mp4"}, {"id": "c163", "say": "Lovely to meet you! Are you a developer or builder, looking for a home, or just exploring?", "lang": "a", "file": "c163.mp4"}, {"id": "c164", "say": "Wonderful! Which company are you with, and what's your role there?", "lang": "a", "file": "c164.mp4"}, {"id": "c165", "say": "Thank you! Shall I give you a quick guided tour of Nexora?", "lang": "a", "file": "c165.mp4"}, {"id": "c166", "say": "Wonderful! Nexora lets you explore homes in 3D, before they're built. Shall I give you a quick guided tour?", "lang": "a", "file": "c166.mp4"}, {"id": "c167", "say": "That's perfectly fine, take your time. Shall I give you a quick guided tour?", "lang": "a", "file": "c167.mp4"}, {"id": "c168", "say": "No problem. You can ask me anything, or explore at your own pace.", "lang": "a", "file": "c168.mp4"}, {"id": "c169", "say": "Let's begin! This is Site Evaluation, where a developer checks the plot before building. Shall we move on?", "lang": "a", "file": "c169.mp4"}, {"id": "c170", "say": "This is Master Planning. The villa layouts are finalised, and the 3D twin goes live for buyers. Shall we move on?", "lang": "a", "file": "c170.mp4"}, {"id": "c171", "say": "Here is Construction. Nexora tracks every milestone with drone images and photo records. Shall we move on?", "lang": "a", "file": "c171.mp4"}, {"id": "c172", "say": "This is Fit-Out and Handover. Every home is quality checked before the keys are handed over. Shall we move on?", "lang": "a", "file": "c172.mp4"}, {"id": "c173", "say": "Welcome to Ready to Move In. The resident app handles bookings, visitors and maintenance. Shall we see the villa?", "lang": "a", "file": "c173.mp4"}, {"id": "c174", "say": "This is the virtual tour of our demo villa. Take your time, and look around.", "lang": "a", "file": "c174.mp4"}, {"id": "c175", "say": "That's the tour! Would you like a personal demo for your own project?", "lang": "a", "file": "c175.mp4"}, {"id": "c176", "say": "No problem, we'll stop here. I'm always in the menu if you need me.", "lang": "a", "file": "c176.mp4"}, {"id": "c177", "say": "I'd be happy to arrange a free demo! What's the best phone number to reach you?", "lang": "a", "file": "c177.mp4"}, {"id": "c178", "say": "I'd be happy to arrange a free demo! May I have your name first?", "lang": "a", "file": "c178.mp4"}, {"id": "c179", "say": "Thank you! What's the best phone number to reach you?", "lang": "a", "file": "c179.mp4"}, {"id": "c180", "say": "Hmm, that number doesn't look right. Could you type it again, with ten digits?", "lang": "a", "file": "c180.mp4"}, {"id": "c181", "say": "Thank you. And your email address?", "lang": "a", "file": "c181.mp4"}, {"id": "c182", "say": "That email doesn't look quite right. Could you type it again?", "lang": "a", "file": "c182.mp4"}, {"id": "c183", "say": "Thank you! Our team will contact you soon. I'm opening the demo form, so you can pick a date.", "lang": "a", "file": "c183.mp4"}, {"id": "c184", "say": "No problem at all. The Book a Demo button is at the top, whenever you're ready.", "lang": "a", "file": "c184.mp4"}, {"id": "n0001", "say": "आराध्या!", "lang": "h", "file": "names/n0001.mp4"}, {"id": "n0002", "say": "आरती!", "lang": "h", "file": "names/n0002.mp4"}, {"id": "n0003", "say": "अदिति!", "lang": "h", "file": "names/n0003.mp4"}, {"id": "n0004", "say": "ऐश्वर्या!", "lang": "h", "file": "names/n0004.mp4"}, {"id": "n0005", "say": "आकांक्षा!", "lang": "h", "file": "names/n0005.mp4"}, {"id": "n0006", "say": "अलका!", "lang": "h", "file": "names/n0006.mp4"}, {"id": "n0007", "say": "अमृता!", "lang": "h", "file": "names/n0007.mp4"}, {"id": "n0008", "say": "अनन्या!", "lang": "h", "file": "names/n0008.mp4"}, {"id": "n0009", "say": "अनीता!", "lang": "h", "file": "names/n0009.mp4"}, {"id": "n0010", "say": "अंजलि!", "lang": "h", "file": "names/n0010.mp4"}, {"id": "n0011", "say": "अंकिता!", "lang": "h", "file": "names/n0011.mp4"}, {"id": "n0012", "say": "अनु!", "lang": "h", "file": "names/n0012.mp4"}, {"id": "n0013", "say": "अनुपमा!", "lang": "h", "file": "names/n0013.mp4"}, {"id": "n0014", "say": "अनुषा!", "lang": "h", "file": "names/n0014.mp4"}, {"id": "n0015", "say": "अपर्णा!", "lang": "h", "file": "names/n0015.mp4"}, {"id": "n0016", "say": "अर्चना!", "lang": "h", "file": "names/n0016.mp4"}, {"id": "n0017", "say": "अर्पिता!", "lang": "h", "file": "names/n0017.mp4"}, {"id": "n0018", "say": "आशा!", "lang": "h", "file": "names/n0018.mp4"}, {"id": "n0019", "say": "अश्विनी!", "lang": "h", "file": "names/n0019.mp4"}, {"id": "n0020", "say": "भावना!", "lang": "h", "file": "names/n0020.mp4"}, {"id": "n0021", "say": "भूमिका!", "lang": "h", "file": "names/n0021.mp4"}, {"id": "n0022", "say": "चांदनी!", "lang": "h", "file": "names/n0022.mp4"}, {"id": "n0023", "say": "चित्रा!", "lang": "h", "file": "names/n0023.mp4"}, {"id": "n0024", "say": "दीपा!", "lang": "h", "file": "names/n0024.mp4"}, {"id": "n0025", "say": "दीपिका!", "lang": "h", "file": "names/n0025.mp4"}, {"id": "n0026", "say": "देविका!", "lang": "h", "file": "names/n0026.mp4"}, {"id": "n0027", "say": "धन्या!", "lang": "h", "file": "names/n0027.mp4"}, {"id": "n0028", "say": "दिशा!", "lang": "h", "file": "names/n0028.mp4"}, {"id": "n0029", "say": "दिव्या!", "lang": "h", "file": "names/n0029.mp4"}, {"id": "n0030", "say": "दुर्गा!", "lang": "h", "file": "names/n0030.mp4"}, {"id": "n0031", "say": "फ़ातिमा!", "lang": "h", "file": "names/n0031.mp4"}, {"id": "n0032", "say": "गौरी!", "lang": "h", "file": "names/n0032.mp4"}, {"id": "n0033", "say": "गायत्री!", "lang": "h", "file": "names/n0033.mp4"}, {"id": "n0034", "say": "गीता!", "lang": "h", "file": "names/n0034.mp4"}, {"id": "n0035", "say": "हेमा!", "lang": "h", "file": "names/n0035.mp4"}, {"id": "n0036", "say": "इंदु!", "lang": "h", "file": "names/n0036.mp4"}, {"id": "n0037", "say": "ईशा!", "lang": "h", "file": "names/n0037.mp4"}, {"id": "n0038", "say": "जया!", "lang": "h", "file": "names/n0038.mp4"}, {"id": "n0039", "say": "ज्योति!", "lang": "h", "file": "names/n0039.mp4"}, {"id": "n0040", "say": "काजल!", "lang": "h", "file": "names/n0040.mp4"}, {"id": "n0041", "say": "कल्पना!", "lang": "h", "file": "names/n0041.mp4"}, {"id": "n0042", "say": "कमला!", "lang": "h", "file": "names/n0042.mp4"}, {"id": "n0043", "say": "कविता!", "lang": "h", "file": "names/n0043.mp4"}, {"id": "n0044", "say": "काव्या!", "lang": "h", "file": "names/n0044.mp4"}, {"id": "n0045", "say": "कीर्ति!", "lang": "h", "file": "names/n0045.mp4"}, {"id": "n0046", "say": "कीर्तना!", "lang": "h", "file": "names/n0046.mp4"}, {"id": "n0047", "say": "कोमल!", "lang": "h", "file": "names/n0047.mp4"}, {"id": "n0048", "say": "कृतिका!", "lang": "h", "file": "names/n0048.mp4"}, {"id": "n0049", "say": "लक्ष्मी!", "lang": "h", "file": "names/n0049.mp4"}, {"id": "n0050", "say": "लता!", "lang": "h", "file": "names/n0050.mp4"}, {"id": "n0051", "say": "माधुरी!", "lang": "h", "file": "names/n0051.mp4"}, {"id": "n0052", "say": "महिमा!", "lang": "h", "file": "names/n0052.mp4"}, {"id": "n0053", "say": "मालिनी!", "lang": "h", "file": "names/n0053.mp4"}, {"id": "n0054", "say": "ममता!", "lang": "h", "file": "names/n0054.mp4"}, {"id": "n0055", "say": "मनीषा!", "lang": "h", "file": "names/n0055.mp4"}, {"id": "n0056", "say": "माया!", "lang": "h", "file": "names/n0056.mp4"}, {"id": "n0057", "say": "मीना!", "lang": "h", "file": "names/n0057.mp4"}, {"id": "n0058", "say": "मीरा!", "lang": "h", "file": "names/n0058.mp4"}, {"id": "n0059", "say": "मेघा!", "lang": "h", "file": "names/n0059.mp4"}, {"id": "n0060", "say": "महक!", "lang": "h", "file": "names/n0060.mp4"}, {"id": "n0061", "say": "मोनिका!", "lang": "h", "file": "names/n0061.mp4"}, {"id": "n0062", "say": "नंदिनी!", "lang": "h", "file": "names/n0062.mp4"}, {"id": "n0063", "say": "नेहा!", "lang": "h", "file": "names/n0063.mp4"}, {"id": "n0064", "say": "निधि!", "lang": "h", "file": "names/n0064.mp4"}, {"id": "n0065", "say": "निकिता!", "lang": "h", "file": "names/n0065.mp4"}, {"id": "n0066", "say": "निशा!", "lang": "h", "file": "names/n0066.mp4"}, {"id": "n0067", "say": "नित्या!", "lang": "h", "file": "names/n0067.mp4"}, {"id": "n0068", "say": "पद्मा!", "lang": "h", "file": "names/n0068.mp4"}, {"id": "n0069", "say": "पल्लवी!", "lang": "h", "file": "names/n0069.mp4"}, {"id": "n0070", "say": "पूजा!", "lang": "h", "file": "names/n0070.mp4"}, {"id": "n0071", "say": "पूनम!", "lang": "h", "file": "names/n0071.mp4"}, {"id": "n0072", "say": "प्राची!", "lang": "h", "file": "names/n0072.mp4"}, {"id": "n0073", "say": "प्रज्ञा!", "lang": "h", "file": "names/n0073.mp4"}, {"id": "n0074", "say": "प्रतिभा!", "lang": "h", "file": "names/n0074.mp4"}, {"id": "n0075", "say": "प्रीति!", "lang": "h", "file": "names/n0075.mp4"}, {"id": "n0076", "say": "प्रिया!", "lang": "h", "file": "names/n0076.mp4"}, {"id": "n0077", "say": "प्रियंका!", "lang": "h", "file": "names/n0077.mp4"}, {"id": "n0078", "say": "राधा!", "lang": "h", "file": "names/n0078.mp4"}, {"id": "n0079", "say": "राधिका!", "lang": "h", "file": "names/n0079.mp4"}, {"id": "n0080", "say": "रानी!", "lang": "h", "file": "names/n0080.mp4"}, {"id": "n0081", "say": "रश्मि!", "lang": "h", "file": "names/n0081.mp4"}, {"id": "n0082", "say": "रेखा!", "lang": "h", "file": "names/n0082.mp4"}, {"id": "n0083", "say": "रेणु!", "lang": "h", "file": "names/n0083.mp4"}, {"id": "n0084", "say": "रेवती!", "lang": "h", "file": "names/n0084.mp4"}, {"id": "n0085", "say": "रिया!", "lang": "h", "file": "names/n0085.mp4"}, {"id": "n0086", "say": "ऋतु!", "lang": "h", "file": "names/n0086.mp4"}, {"id": "n0087", "say": "रोहिणी!", "lang": "h", "file": "names/n0087.mp4"}, {"id": "n0088", "say": "रुचि!", "lang": "h", "file": "names/n0088.mp4"}, {"id": "n0089", "say": "रूपा!", "lang": "h", "file": "names/n0089.mp4"}, {"id": "n0090", "say": "साक्षी!", "lang": "h", "file": "names/n0090.mp4"}, {"id": "n0091", "say": "संगीता!", "lang": "h", "file": "names/n0091.mp4"}, {"id": "n0092", "say": "संजना!", "lang": "h", "file": "names/n0092.mp4"}, {"id": "n0093", "say": "सपना!", "lang": "h", "file": "names/n0093.mp4"}, {"id": "n0094", "say": "सारा!", "lang": "h", "file": "names/n0094.mp4"}, {"id": "n0095", "say": "सरिता!", "lang": "h", "file": "names/n0095.mp4"}, {"id": "n0096", "say": "सविता!", "lang": "h", "file": "names/n0096.mp4"}, {"id": "n0097", "say": "सीमा!", "lang": "h", "file": "names/n0097.mp4"}, {"id": "n0098", "say": "शालिनी!", "lang": "h", "file": "names/n0098.mp4"}, {"id": "n0099", "say": "शांति!", "lang": "h", "file": "names/n0099.mp4"}, {"id": "n0100", "say": "शर्मिला!", "lang": "h", "file": "names/n0100.mp4"}, {"id": "n0101", "say": "शिल्पा!", "lang": "h", "file": "names/n0101.mp4"}, {"id": "n0102", "say": "शोभा!", "lang": "h", "file": "names/n0102.mp4"}, {"id": "n0103", "say": "श्रेया!", "lang": "h", "file": "names/n0103.mp4"}, {"id": "n0104", "say": "श्रुति!", "lang": "h", "file": "names/n0104.mp4"}, {"id": "n0105", "say": "श्वेता!", "lang": "h", "file": "names/n0105.mp4"}, {"id": "n0106", "say": "सिमरन!", "lang": "h", "file": "names/n0106.mp4"}, {"id": "n0107", "say": "स्मिता!", "lang": "h", "file": "names/n0107.mp4"}, {"id": "n0108", "say": "स्नेहा!", "lang": "h", "file": "names/n0108.mp4"}, {"id": "n0109", "say": "सोनल!", "lang": "h", "file": "names/n0109.mp4"}, {"id": "n0110", "say": "सोनिया!", "lang": "h", "file": "names/n0110.mp4"}, {"id": "n0111", "say": "सुमन!", "lang": "h", "file": "names/n0111.mp4"}, {"id": "n0112", "say": "सुनीता!", "lang": "h", "file": "names/n0112.mp4"}, {"id": "n0113", "say": "सुप्रिया!", "lang": "h", "file": "names/n0113.mp4"}, {"id": "n0114", "say": "सुषमा!", "lang": "h", "file": "names/n0114.mp4"}, {"id": "n0115", "say": "स्वाति!", "lang": "h", "file": "names/n0115.mp4"}, {"id": "n0116", "say": "तन्वी!", "lang": "h", "file": "names/n0116.mp4"}, {"id": "n0117", "say": "तान्या!", "lang": "h", "file": "names/n0117.mp4"}, {"id": "n0118", "say": "उमा!", "lang": "h", "file": "names/n0118.mp4"}, {"id": "n0119", "say": "उषा!", "lang": "h", "file": "names/n0119.mp4"}, {"id": "n0120", "say": "वैष्णवी!", "lang": "h", "file": "names/n0120.mp4"}, {"id": "n0121", "say": "वंदना!", "lang": "h", "file": "names/n0121.mp4"}, {"id": "n0122", "say": "वर्षा!", "lang": "h", "file": "names/n0122.mp4"}, {"id": "n0123", "say": "विद्या!", "lang": "h", "file": "names/n0123.mp4"}, {"id": "n0124", "say": "विजया!", "lang": "h", "file": "names/n0124.mp4"}, {"id": "n0125", "say": "विनीता!", "lang": "h", "file": "names/n0125.mp4"}, {"id": "n0126", "say": "यामिनी!", "lang": "h", "file": "names/n0126.mp4"}, {"id": "n0127", "say": "ज़ारा!", "lang": "h", "file": "names/n0127.mp4"}, {"id": "n0128", "say": "ज़ोया!", "lang": "h", "file": "names/n0128.mp4"}, {"id": "n0129", "say": "आयशा!", "lang": "h", "file": "names/n0129.mp4"}, {"id": "n0130", "say": "सना!", "lang": "h", "file": "names/n0130.mp4"}, {"id": "n0131", "say": "नाज़िया!", "lang": "h", "file": "names/n0131.mp4"}, {"id": "n0132", "say": "शबनम!", "lang": "h", "file": "names/n0132.mp4"}, {"id": "n0133", "say": "फ़राह!", "lang": "h", "file": "names/n0133.mp4"}, {"id": "n0134", "say": "हिना!", "lang": "h", "file": "names/n0134.mp4"}, {"id": "n0135", "say": "मुमताज़!", "lang": "h", "file": "names/n0135.mp4"}, {"id": "n0136", "say": "नूर!", "lang": "h", "file": "names/n0136.mp4"}, {"id": "n0137", "say": "हरप्रीत!", "lang": "h", "file": "names/n0137.mp4"}, {"id": "n0138", "say": "गुरप्रीत!", "lang": "h", "file": "names/n0138.mp4"}, {"id": "n0139", "say": "जसप्रीत!", "lang": "h", "file": "names/n0139.mp4"}, {"id": "n0140", "say": "मनप्रीत!", "lang": "h", "file": "names/n0140.mp4"}, {"id": "n0141", "say": "नवनीत!", "lang": "h", "file": "names/n0141.mp4"}, {"id": "n0142", "say": "किरण!", "lang": "h", "file": "names/n0142.mp4"}, {"id": "n0143", "say": "परमिंदर!", "lang": "h", "file": "names/n0143.mp4"}, {"id": "n0144", "say": "आरोही!", "lang": "h", "file": "names/n0144.mp4"}, {"id": "n0145", "say": "अन्वी!", "lang": "h", "file": "names/n0145.mp4"}, {"id": "n0146", "say": "अवनी!", "lang": "h", "file": "names/n0146.mp4"}, {"id": "n0147", "say": "दिया!", "lang": "h", "file": "names/n0147.mp4"}, {"id": "n0148", "say": "इरा!", "lang": "h", "file": "names/n0148.mp4"}, {"id": "n0149", "say": "इशिता!", "lang": "h", "file": "names/n0149.mp4"}, {"id": "n0150", "say": "जाह्नवी!", "lang": "h", "file": "names/n0150.mp4"}, {"id": "n0151", "say": "ख़ुशी!", "lang": "h", "file": "names/n0151.mp4"}, {"id": "n0152", "say": "कृति!", "lang": "h", "file": "names/n0152.mp4"}, {"id": "n0153", "say": "मानसी!", "lang": "h", "file": "names/n0153.mp4"}, {"id": "n0154", "say": "मायरा!", "lang": "h", "file": "names/n0154.mp4"}, {"id": "n0155", "say": "नव्या!", "lang": "h", "file": "names/n0155.mp4"}, {"id": "n0156", "say": "परी!", "lang": "h", "file": "names/n0156.mp4"}, {"id": "n0157", "say": "प्रिशा!", "lang": "h", "file": "names/n0157.mp4"}, {"id": "n0158", "say": "सान्वी!", "lang": "h", "file": "names/n0158.mp4"}, {"id": "n0159", "say": "शनाया!", "lang": "h", "file": "names/n0159.mp4"}, {"id": "n0160", "say": "सिया!", "lang": "h", "file": "names/n0160.mp4"}, {"id": "n0161", "say": "तारा!", "lang": "h", "file": "names/n0161.mp4"}, {"id": "n0162", "say": "तृषा!", "lang": "h", "file": "names/n0162.mp4"}, {"id": "n0163", "say": "वान्या!", "lang": "h", "file": "names/n0163.mp4"}, {"id": "n0164", "say": "भव्या!", "lang": "h", "file": "names/n0164.mp4"}, {"id": "n0165", "say": "चारु!", "lang": "h", "file": "names/n0165.mp4"}, {"id": "n0166", "say": "एकता!", "lang": "h", "file": "names/n0166.mp4"}, {"id": "n0167", "say": "गरिमा!", "lang": "h", "file": "names/n0167.mp4"}, {"id": "n0168", "say": "कनिका!", "lang": "h", "file": "names/n0168.mp4"}, {"id": "n0169", "say": "लावण्या!", "lang": "h", "file": "names/n0169.mp4"}, {"id": "n0170", "say": "मिताली!", "lang": "h", "file": "names/n0170.mp4"}, {"id": "n0171", "say": "नम्रता!", "lang": "h", "file": "names/n0171.mp4"}, {"id": "n0172", "say": "पायल!", "lang": "h", "file": "names/n0172.mp4"}, {"id": "n0173", "say": "रचना!", "lang": "h", "file": "names/n0173.mp4"}, {"id": "n0174", "say": "राखी!", "lang": "h", "file": "names/n0174.mp4"}, {"id": "n0175", "say": "रीना!", "lang": "h", "file": "names/n0175.mp4"}, {"id": "n0176", "say": "साधना!", "lang": "h", "file": "names/n0176.mp4"}, {"id": "n0177", "say": "शारदा!", "lang": "h", "file": "names/n0177.mp4"}, {"id": "n0178", "say": "सुधा!", "lang": "h", "file": "names/n0178.mp4"}, {"id": "n0179", "say": "सुजाता!", "lang": "h", "file": "names/n0179.mp4"}, {"id": "n0180", "say": "सुनंदा!", "lang": "h", "file": "names/n0180.mp4"}, {"id": "n0181", "say": "तेजस्विनी!", "lang": "h", "file": "names/n0181.mp4"}, {"id": "n0182", "say": "उर्मिला!", "lang": "h", "file": "names/n0182.mp4"}, {"id": "n0183", "say": "बिंदु!", "lang": "h", "file": "names/n0183.mp4"}, {"id": "n0184", "say": "शोभना!", "lang": "h", "file": "names/n0184.mp4"}, {"id": "n0185", "say": "सौम्या!", "lang": "h", "file": "names/n0185.mp4"}, {"id": "n0186", "say": "रम्या!", "lang": "h", "file": "names/n0186.mp4"}, {"id": "n0187", "say": "हरिणी!", "lang": "h", "file": "names/n0187.mp4"}, {"id": "n0188", "say": "जननी!", "lang": "h", "file": "names/n0188.mp4"}, {"id": "n0189", "say": "माधवी!", "lang": "h", "file": "names/n0189.mp4"}, {"id": "n0190", "say": "नलिनी!", "lang": "h", "file": "names/n0190.mp4"}, {"id": "n0191", "say": "पवित्रा!", "lang": "h", "file": "names/n0191.mp4"}, {"id": "n0192", "say": "रंजनी!", "lang": "h", "file": "names/n0192.mp4"}, {"id": "n0193", "say": "शरण्या!", "lang": "h", "file": "names/n0193.mp4"}, {"id": "n0194", "say": "सिंधु!", "lang": "h", "file": "names/n0194.mp4"}, {"id": "n0195", "say": "सौजन्या!", "lang": "h", "file": "names/n0195.mp4"}, {"id": "n0196", "say": "सुमति!", "lang": "h", "file": "names/n0196.mp4"}, {"id": "n0197", "say": "सुगन्या!", "lang": "h", "file": "names/n0197.mp4"}, {"id": "n0198", "say": "स्वप्ना!", "lang": "h", "file": "names/n0198.mp4"}, {"id": "n0199", "say": "वसंती!", "lang": "h", "file": "names/n0199.mp4"}, {"id": "n0200", "say": "वाणी!", "lang": "h", "file": "names/n0200.mp4"}, {"id": "n0201", "say": "यमुना!", "lang": "h", "file": "names/n0201.mp4"}, {"id": "n0202", "say": "अंबिका!", "lang": "h", "file": "names/n0202.mp4"}, {"id": "n0203", "say": "चैत्रा!", "lang": "h", "file": "names/n0203.mp4"}, {"id": "n0204", "say": "दीप्ति!", "lang": "h", "file": "names/n0204.mp4"}, {"id": "n0205", "say": "हेमलता!", "lang": "h", "file": "names/n0205.mp4"}, {"id": "n0206", "say": "जयश्री!", "lang": "h", "file": "names/n0206.mp4"}, {"id": "n0207", "say": "कुसुमा!", "lang": "h", "file": "names/n0207.mp4"}, {"id": "n0208", "say": "मंजुला!", "lang": "h", "file": "names/n0208.mp4"}, {"id": "n0209", "say": "पार्वती!", "lang": "h", "file": "names/n0209.mp4"}, {"id": "n0210", "say": "पुष्पा!", "lang": "h", "file": "names/n0210.mp4"}, {"id": "n0211", "say": "राजेश्वरी!", "lang": "h", "file": "names/n0211.mp4"}, {"id": "n0212", "say": "रुक्मिणी!", "lang": "h", "file": "names/n0212.mp4"}, {"id": "n0213", "say": "श्रीदेवी!", "lang": "h", "file": "names/n0213.mp4"}, {"id": "n0214", "say": "तुलसी!", "lang": "h", "file": "names/n0214.mp4"}, {"id": "n0215", "say": "वीणा!", "lang": "h", "file": "names/n0215.mp4"}, {"id": "n0216", "say": "भवानी!", "lang": "h", "file": "names/n0216.mp4"}, {"id": "n0217", "say": "शीला!", "lang": "h", "file": "names/n0217.mp4"}, {"id": "n0218", "say": "लीला!", "lang": "h", "file": "names/n0218.mp4"}, {"id": "n0219", "say": "रेशमा!", "lang": "h", "file": "names/n0219.mp4"}, {"id": "n0220", "say": "श्रीजा!", "lang": "h", "file": "names/n0220.mp4"}, {"id": "n0221", "say": "आतिरा!", "lang": "h", "file": "names/n0221.mp4"}, {"id": "n0222", "say": "अश्वती!", "lang": "h", "file": "names/n0222.mp4"}, {"id": "n0223", "say": "गोपिका!", "lang": "h", "file": "names/n0223.mp4"}, {"id": "n0224", "say": "अंजू!", "lang": "h", "file": "names/n0224.mp4"}, {"id": "n0225", "say": "लेखा!", "lang": "h", "file": "names/n0225.mp4"}, {"id": "n0226", "say": "नीतू!", "lang": "h", "file": "names/n0226.mp4"}, {"id": "n0227", "say": "रेम्या!", "lang": "h", "file": "names/n0227.mp4"}, {"id": "n0228", "say": "देबजानी!", "lang": "h", "file": "names/n0228.mp4"}, {"id": "n0229", "say": "मौमिता!", "lang": "h", "file": "names/n0229.mp4"}, {"id": "n0230", "say": "ऋतुपर्णा!", "lang": "h", "file": "names/n0230.mp4"}, {"id": "n0231", "say": "शर्मिष्ठा!", "lang": "h", "file": "names/n0231.mp4"}, {"id": "n0232", "say": "सोहिनी!", "lang": "h", "file": "names/n0232.mp4"}, {"id": "n0233", "say": "सुदेष्णा!", "lang": "h", "file": "names/n0233.mp4"}, {"id": "n0234", "say": "तनुश्री!", "lang": "h", "file": "names/n0234.mp4"}, {"id": "n0235", "say": "तनीषा!", "lang": "h", "file": "names/n0235.mp4"}, {"id": "n0236", "say": "मौसमी!", "lang": "h", "file": "names/n0236.mp4"}, {"id": "n0237", "say": "रूपाली!", "lang": "h", "file": "names/n0237.mp4"}, {"id": "n0238", "say": "अरुंधति!", "lang": "h", "file": "names/n0238.mp4"}, {"id": "n0239", "say": "इप्सिता!", "lang": "h", "file": "names/n0239.mp4"}, {"id": "n0240", "say": "सुभश्री!", "lang": "h", "file": "names/n0240.mp4"}, {"id": "n0241", "say": "आर्या!", "lang": "h", "file": "names/n0241.mp4"}, {"id": "n0242", "say": "केतकी!", "lang": "h", "file": "names/n0242.mp4"}, {"id": "n0243", "say": "मृणाल!", "lang": "h", "file": "names/n0243.mp4"}, {"id": "n0244", "say": "ऋतुजा!", "lang": "h", "file": "names/n0244.mp4"}, {"id": "n0245", "say": "सायली!", "lang": "h", "file": "names/n0245.mp4"}, {"id": "n0246", "say": "श्रद्धा!", "lang": "h", "file": "names/n0246.mp4"}, {"id": "n0247", "say": "स्नेहल!", "lang": "h", "file": "names/n0247.mp4"}, {"id": "n0248", "say": "तेजल!", "lang": "h", "file": "names/n0248.mp4"}, {"id": "n0249", "say": "हेतल!", "lang": "h", "file": "names/n0249.mp4"}, {"id": "n0250", "say": "कृपा!", "lang": "h", "file": "names/n0250.mp4"}, {"id": "n0251", "say": "निराली!", "lang": "h", "file": "names/n0251.mp4"}, {"id": "n0252", "say": "धारा!", "lang": "h", "file": "names/n0252.mp4"}, {"id": "n0253", "say": "जिनल!", "lang": "h", "file": "names/n0253.mp4"}, {"id": "n0254", "say": "ख़ुशबू!", "lang": "h", "file": "names/n0254.mp4"}, {"id": "n0255", "say": "उर्वी!", "lang": "h", "file": "names/n0255.mp4"}, {"id": "n0256", "say": "भक्ति!", "lang": "h", "file": "names/n0256.mp4"}, {"id": "n0257", "say": "वृषाली!", "lang": "h", "file": "names/n0257.mp4"}, {"id": "n0258", "say": "मधुरा!", "lang": "h", "file": "names/n0258.mp4"}, {"id": "n0259", "say": "आफ़रीन!", "lang": "h", "file": "names/n0259.mp4"}, {"id": "n0260", "say": "अलीशा!", "lang": "h", "file": "names/n0260.mp4"}, {"id": "n0261", "say": "अमीना!", "lang": "h", "file": "names/n0261.mp4"}, {"id": "n0262", "say": "अस्मा!", "lang": "h", "file": "names/n0262.mp4"}, {"id": "n0263", "say": "फ़रज़ाना!", "lang": "h", "file": "names/n0263.mp4"}, {"id": "n0264", "say": "हुमा!", "lang": "h", "file": "names/n0264.mp4"}, {"id": "n0265", "say": "इक़रा!", "lang": "h", "file": "names/n0265.mp4"}, {"id": "n0266", "say": "मेहरीन!", "lang": "h", "file": "names/n0266.mp4"}, {"id": "n0267", "say": "नरगिस!", "lang": "h", "file": "names/n0267.mp4"}, {"id": "n0268", "say": "परवीन!", "lang": "h", "file": "names/n0268.mp4"}, {"id": "n0269", "say": "रुबीना!", "lang": "h", "file": "names/n0269.mp4"}, {"id": "n0270", "say": "रुख़साना!", "lang": "h", "file": "names/n0270.mp4"}, {"id": "n0271", "say": "सबा!", "lang": "h", "file": "names/n0271.mp4"}, {"id": "n0272", "say": "सादिया!", "lang": "h", "file": "names/n0272.mp4"}, {"id": "n0273", "say": "सलमा!", "lang": "h", "file": "names/n0273.mp4"}, {"id": "n0274", "say": "समीना!", "lang": "h", "file": "names/n0274.mp4"}, {"id": "n0275", "say": "शबाना!", "lang": "h", "file": "names/n0275.mp4"}, {"id": "n0276", "say": "शाज़िया!", "lang": "h", "file": "names/n0276.mp4"}, {"id": "n0277", "say": "तबस्सुम!", "lang": "h", "file": "names/n0277.mp4"}, {"id": "n0278", "say": "यास्मीन!", "lang": "h", "file": "names/n0278.mp4"}, {"id": "n0279", "say": "ज़ैनब!", "lang": "h", "file": "names/n0279.mp4"}, {"id": "n0280", "say": "ज़ीनत!", "lang": "h", "file": "names/n0280.mp4"}, {"id": "n0281", "say": "उज़्मा!", "lang": "h", "file": "names/n0281.mp4"}, {"id": "n0282", "say": "नुसरत!", "lang": "h", "file": "names/n0282.mp4"}, {"id": "n0283", "say": "सानिया!", "lang": "h", "file": "names/n0283.mp4"}, {"id": "n0284", "say": "आरव!", "lang": "h", "file": "names/n0284.mp4"}, {"id": "n0285", "say": "अभय!", "lang": "h", "file": "names/n0285.mp4"}, {"id": "n0286", "say": "अभिनव!", "lang": "h", "file": "names/n0286.mp4"}, {"id": "n0287", "say": "अभिषेक!", "lang": "h", "file": "names/n0287.mp4"}, {"id": "n0288", "say": "आदित्य!", "lang": "h", "file": "names/n0288.mp4"}, {"id": "n0289", "say": "अजय!", "lang": "h", "file": "names/n0289.mp4"}, {"id": "n0290", "say": "अजीत!", "lang": "h", "file": "names/n0290.mp4"}, {"id": "n0291", "say": "आकाश!", "lang": "h", "file": "names/n0291.mp4"}, {"id": "n0292", "say": "अखिल!", "lang": "h", "file": "names/n0292.mp4"}, {"id": "n0293", "say": "अक्षय!", "lang": "h", "file": "names/n0293.mp4"}, {"id": "n0294", "say": "आलोक!", "lang": "h", "file": "names/n0294.mp4"}, {"id": "n0295", "say": "अमन!", "lang": "h", "file": "names/n0295.mp4"}, {"id": "n0296", "say": "अमर!", "lang": "h", "file": "names/n0296.mp4"}, {"id": "n0297", "say": "अमित!", "lang": "h", "file": "names/n0297.mp4"}, {"id": "n0298", "say": "अमोल!", "lang": "h", "file": "names/n0298.mp4"}, {"id": "n0299", "say": "आनंद!", "lang": "h", "file": "names/n0299.mp4"}, {"id": "n0300", "say": "अनिल!", "lang": "h", "file": "names/n0300.mp4"}, {"id": "n0301", "say": "अनिकेत!", "lang": "h", "file": "names/n0301.mp4"}, {"id": "n0302", "say": "अनिरुद्ध!", "lang": "h", "file": "names/n0302.mp4"}, {"id": "n0303", "say": "अंकित!", "lang": "h", "file": "names/n0303.mp4"}, {"id": "n0304", "say": "अंकुर!", "lang": "h", "file": "names/n0304.mp4"}, {"id": "n0305", "say": "अनूप!", "lang": "h", "file": "names/n0305.mp4"}, {"id": "n0306", "say": "अंशुल!", "lang": "h", "file": "names/n0306.mp4"}, {"id": "n0307", "say": "अनुज!", "lang": "h", "file": "names/n0307.mp4"}, {"id": "n0308", "say": "अर्जुन!", "lang": "h", "file": "names/n0308.mp4"}, {"id": "n0309", "say": "अर्णव!", "lang": "h", "file": "names/n0309.mp4"}, {"id": "n0310", "say": "अरुण!", "lang": "h", "file": "names/n0310.mp4"}, {"id": "n0311", "say": "अरविंद!", "lang": "h", "file": "names/n0311.mp4"}, {"id": "n0312", "say": "आशीष!", "lang": "h", "file": "names/n0312.mp4"}, {"id": "n0313", "say": "अशोक!", "lang": "h", "file": "names/n0313.mp4"}, {"id": "n0314", "say": "अश्विन!", "lang": "h", "file": "names/n0314.mp4"}, {"id": "n0315", "say": "अतुल!", "lang": "h", "file": "names/n0315.mp4"}, {"id": "n0316", "say": "आयुष!", "lang": "h", "file": "names/n0316.mp4"}, {"id": "n0317", "say": "बालाजी!", "lang": "h", "file": "names/n0317.mp4"}, {"id": "n0318", "say": "भरत!", "lang": "h", "file": "names/n0318.mp4"}, {"id": "n0319", "say": "भास्कर!", "lang": "h", "file": "names/n0319.mp4"}, {"id": "n0320", "say": "चंदन!", "lang": "h", "file": "names/n0320.mp4"}, {"id": "n0321", "say": "चेतन!", "lang": "h", "file": "names/n0321.mp4"}, {"id": "n0322", "say": "दीपक!", "lang": "h", "file": "names/n0322.mp4"}, {"id": "n0323", "say": "देव!", "lang": "h", "file": "names/n0323.mp4"}, {"id": "n0324", "say": "देवेंद्र!", "lang": "h", "file": "names/n0324.mp4"}, {"id": "n0325", "say": "ध्रुव!", "lang": "h", "file": "names/n0325.mp4"}, {"id": "n0326", "say": "दिनेश!", "lang": "h", "file": "names/n0326.mp4"}, {"id": "n0327", "say": "गणेश!", "lang": "h", "file": "names/n0327.mp4"}, {"id": "n0328", "say": "गौरव!", "lang": "h", "file": "names/n0328.mp4"}, {"id": "n0329", "say": "गिरीश!", "lang": "h", "file": "names/n0329.mp4"}, {"id": "n0330", "say": "गोपाल!", "lang": "h", "file": "names/n0330.mp4"}, {"id": "n0331", "say": "गौतम!", "lang": "h", "file": "names/n0331.mp4"}, {"id": "n0332", "say": "हरि!", "lang": "h", "file": "names/n0332.mp4"}, {"id": "n0333", "say": "हरीश!", "lang": "h", "file": "names/n0333.mp4"}, {"id": "n0334", "say": "हेमंत!", "lang": "h", "file": "names/n0334.mp4"}, {"id": "n0335", "say": "हितेश!", "lang": "h", "file": "names/n0335.mp4"}, {"id": "n0336", "say": "ईशान!", "lang": "h", "file": "names/n0336.mp4"}, {"id": "n0337", "say": "जगदीश!", "lang": "h", "file": "names/n0337.mp4"}, {"id": "n0338", "say": "जतिन!", "lang": "h", "file": "names/n0338.mp4"}, {"id": "n0339", "say": "जय!", "lang": "h", "file": "names/n0339.mp4"}, {"id": "n0340", "say": "जयंत!", "lang": "h", "file": "names/n0340.mp4"}, {"id": "n0341", "say": "जितेंद्र!", "lang": "h", "file": "names/n0341.mp4"}, {"id": "n0342", "say": "कमल!", "lang": "h", "file": "names/n0342.mp4"}, {"id": "n0343", "say": "करण!", "lang": "h", "file": "names/n0343.mp4"}, {"id": "n0344", "say": "कार्तिक!", "lang": "h", "file": "names/n0344.mp4"}, {"id": "n0345", "say": "किशोर!", "lang": "h", "file": "names/n0345.mp4"}, {"id": "n0346", "say": "कृष्णा!", "lang": "h", "file": "names/n0346.mp4"}, {"id": "n0347", "say": "कुणाल!", "lang": "h", "file": "names/n0347.mp4"}, {"id": "n0348", "say": "ललित!", "lang": "h", "file": "names/n0348.mp4"}, {"id": "n0349", "say": "लोकेश!", "lang": "h", "file": "names/n0349.mp4"}, {"id": "n0350", "say": "माधव!", "lang": "h", "file": "names/n0350.mp4"}, {"id": "n0351", "say": "महेश!", "lang": "h", "file": "names/n0351.mp4"}, {"id": "n0352", "say": "मनीष!", "lang": "h", "file": "names/n0352.mp4"}, {"id": "n0353", "say": "मनोज!", "lang": "h", "file": "names/n0353.mp4"}, {"id": "n0354", "say": "मयंक!", "lang": "h", "file": "names/n0354.mp4"}, {"id": "n0355", "say": "मोहन!", "lang": "h", "file": "names/n0355.mp4"}, {"id": "n0356", "say": "मुकेश!", "lang": "h", "file": "names/n0356.mp4"}, {"id": "n0357", "say": "मुरली!", "lang": "h", "file": "names/n0357.mp4"}, {"id": "n0358", "say": "नरेश!", "lang": "h", "file": "names/n0358.mp4"}, {"id": "n0359", "say": "नवीन!", "lang": "h", "file": "names/n0359.mp4"}, {"id": "n0360", "say": "नीरज!", "lang": "h", "file": "names/n0360.mp4"}, {"id": "n0361", "say": "निखिल!", "lang": "h", "file": "names/n0361.mp4"}, {"id": "n0362", "say": "नीलेश!", "lang": "h", "file": "names/n0362.mp4"}, {"id": "n0363", "say": "नितिन!", "lang": "h", "file": "names/n0363.mp4"}, {"id": "n0364", "say": "पंकज!", "lang": "h", "file": "names/n0364.mp4"}, {"id": "n0365", "say": "पारस!", "lang": "h", "file": "names/n0365.mp4"}, {"id": "n0366", "say": "पवन!", "lang": "h", "file": "names/n0366.mp4"}, {"id": "n0367", "say": "पीयूष!", "lang": "h", "file": "names/n0367.mp4"}, {"id": "n0368", "say": "प्रदीप!", "lang": "h", "file": "names/n0368.mp4"}, {"id": "n0369", "say": "प्रकाश!", "lang": "h", "file": "names/n0369.mp4"}, {"id": "n0370", "say": "प्रणव!", "lang": "h", "file": "names/n0370.mp4"}, {"id": "n0371", "say": "प्रसाद!", "lang": "h", "file": "names/n0371.mp4"}, {"id": "n0372", "say": "प्रशांत!", "lang": "h", "file": "names/n0372.mp4"}, {"id": "n0373", "say": "प्रवीण!", "lang": "h", "file": "names/n0373.mp4"}, {"id": "n0374", "say": "प्रेम!", "lang": "h", "file": "names/n0374.mp4"}, {"id": "n0375", "say": "पुनीत!", "lang": "h", "file": "names/n0375.mp4"}, {"id": "n0376", "say": "राहुल!", "lang": "h", "file": "names/n0376.mp4"}, {"id": "n0377", "say": "राज!", "lang": "h", "file": "names/n0377.mp4"}, {"id": "n0378", "say": "रजत!", "lang": "h", "file": "names/n0378.mp4"}, {"id": "n0379", "say": "राजेश!", "lang": "h", "file": "names/n0379.mp4"}, {"id": "n0380", "say": "राजीव!", "lang": "h", "file": "names/n0380.mp4"}, {"id": "n0381", "say": "राकेश!", "lang": "h", "file": "names/n0381.mp4"}, {"id": "n0382", "say": "राम!", "lang": "h", "file": "names/n0382.mp4"}, {"id": "n0383", "say": "रमेश!", "lang": "h", "file": "names/n0383.mp4"}, {"id": "n0384", "say": "रंजीत!", "lang": "h", "file": "names/n0384.mp4"}, {"id": "n0385", "say": "रवि!", "lang": "h", "file": "names/n0385.mp4"}, {"id": "n0386", "say": "ऋषि!", "lang": "h", "file": "names/n0386.mp4"}, {"id": "n0387", "say": "रितेश!", "lang": "h", "file": "names/n0387.mp4"}, {"id": "n0388", "say": "रोहन!", "lang": "h", "file": "names/n0388.mp4"}, {"id": "n0389", "say": "रोहित!", "lang": "h", "file": "names/n0389.mp4"}, {"id": "n0390", "say": "सचिन!", "lang": "h", "file": "names/n0390.mp4"}, {"id": "n0391", "say": "सागर!", "lang": "h", "file": "names/n0391.mp4"}, {"id": "n0392", "say": "साहिल!", "lang": "h", "file": "names/n0392.mp4"}, {"id": "n0393", "say": "समीर!", "lang": "h", "file": "names/n0393.mp4"}, {"id": "n0394", "say": "संदीप!", "lang": "h", "file": "names/n0394.mp4"}, {"id": "n0395", "say": "संजय!", "lang": "h", "file": "names/n0395.mp4"}, {"id": "n0396", "say": "संतोष!", "lang": "h", "file": "names/n0396.mp4"}, {"id": "n0397", "say": "शरत!", "lang": "h", "file": "names/n0397.mp4"}, {"id": "n0398", "say": "सौरभ!", "lang": "h", "file": "names/n0398.mp4"}, {"id": "n0399", "say": "शंकर!", "lang": "h", "file": "names/n0399.mp4"}, {"id": "n0400", "say": "शरण!", "lang": "h", "file": "names/n0400.mp4"}, {"id": "n0401", "say": "चरण!", "lang": "h", "file": "names/n0401.mp4"}, {"id": "n0402", "say": "शिव!", "lang": "h", "file": "names/n0402.mp4"}, {"id": "n0403", "say": "शिवम!", "lang": "h", "file": "names/n0403.mp4"}, {"id": "n0404", "say": "शुभम!", "lang": "h", "file": "names/n0404.mp4"}, {"id": "n0405", "say": "सिद्धार्थ!", "lang": "h", "file": "names/n0405.mp4"}, {"id": "n0406", "say": "सोनू!", "lang": "h", "file": "names/n0406.mp4"}, {"id": "n0407", "say": "सौरव!", "lang": "h", "file": "names/n0407.mp4"}, {"id": "n0408", "say": "श्रीनिवास!", "lang": "h", "file": "names/n0408.mp4"}, {"id": "n0409", "say": "सुभाष!", "lang": "h", "file": "names/n0409.mp4"}, {"id": "n0410", "say": "सुधीर!", "lang": "h", "file": "names/n0410.mp4"}, {"id": "n0411", "say": "सुमित!", "lang": "h", "file": "names/n0411.mp4"}, {"id": "n0412", "say": "सुनील!", "lang": "h", "file": "names/n0412.mp4"}, {"id": "n0413", "say": "सूरज!", "lang": "h", "file": "names/n0413.mp4"}, {"id": "n0414", "say": "सुरेश!", "lang": "h", "file": "names/n0414.mp4"}, {"id": "n0415", "say": "तरुण!", "lang": "h", "file": "names/n0415.mp4"}, {"id": "n0416", "say": "तुषार!", "lang": "h", "file": "names/n0416.mp4"}, {"id": "n0417", "say": "उदय!", "lang": "h", "file": "names/n0417.mp4"}, {"id": "n0418", "say": "उमेश!", "lang": "h", "file": "names/n0418.mp4"}, {"id": "n0419", "say": "वरुण!", "lang": "h", "file": "names/n0419.mp4"}, {"id": "n0420", "say": "विजय!", "lang": "h", "file": "names/n0420.mp4"}, {"id": "n0421", "say": "विकास!", "lang": "h", "file": "names/n0421.mp4"}, {"id": "n0422", "say": "विक्रम!", "lang": "h", "file": "names/n0422.mp4"}, {"id": "n0423", "say": "विनय!", "lang": "h", "file": "names/n0423.mp4"}, {"id": "n0424", "say": "विनायक!", "lang": "h", "file": "names/n0424.mp4"}, {"id": "n0425", "say": "विनोद!", "lang": "h", "file": "names/n0425.mp4"}, {"id": "n0426", "say": "विपिन!", "lang": "h", "file": "names/n0426.mp4"}, {"id": "n0427", "say": "विशाल!", "lang": "h", "file": "names/n0427.mp4"}, {"id": "n0428", "say": "विवेक!", "lang": "h", "file": "names/n0428.mp4"}, {"id": "n0429", "say": "यश!", "lang": "h", "file": "names/n0429.mp4"}, {"id": "n0430", "say": "योगेश!", "lang": "h", "file": "names/n0430.mp4"}, {"id": "n0431", "say": "बालू!", "lang": "h", "file": "names/n0431.mp4"}, {"id": "n0432", "say": "धनुष!", "lang": "h", "file": "names/n0432.mp4"}, {"id": "n0433", "say": "गणपति!", "lang": "h", "file": "names/n0433.mp4"}, {"id": "n0434", "say": "गोपी!", "lang": "h", "file": "names/n0434.mp4"}, {"id": "n0435", "say": "हरिहरन!", "lang": "h", "file": "names/n0435.mp4"}, {"id": "n0436", "say": "जगन!", "lang": "h", "file": "names/n0436.mp4"}, {"id": "n0437", "say": "कार्तिकेयन!", "lang": "h", "file": "names/n0437.mp4"}, {"id": "n0438", "say": "कुमार!", "lang": "h", "file": "names/n0438.mp4"}, {"id": "n0439", "say": "मदन!", "lang": "h", "file": "names/n0439.mp4"}, {"id": "n0440", "say": "मणिकंदन!", "lang": "h", "file": "names/n0440.mp4"}, {"id": "n0441", "say": "मुरुगन!", "lang": "h", "file": "names/n0441.mp4"}, {"id": "n0442", "say": "मुत्तु!", "lang": "h", "file": "names/n0442.mp4"}, {"id": "n0443", "say": "प्रभु!", "lang": "h", "file": "names/n0443.mp4"}, {"id": "n0444", "say": "राघव!", "lang": "h", "file": "names/n0444.mp4"}, {"id": "n0445", "say": "रघु!", "lang": "h", "file": "names/n0445.mp4"}, {"id": "n0446", "say": "राजकुमार!", "lang": "h", "file": "names/n0446.mp4"}, {"id": "n0447", "say": "सरवणन!", "lang": "h", "file": "names/n0447.mp4"}, {"id": "n0448", "say": "सेंथिल!", "lang": "h", "file": "names/n0448.mp4"}, {"id": "n0449", "say": "शिवा!", "lang": "h", "file": "names/n0449.mp4"}, {"id": "n0450", "say": "श्रीधर!", "lang": "h", "file": "names/n0450.mp4"}, {"id": "n0451", "say": "श्रीकांत!", "lang": "h", "file": "names/n0451.mp4"}, {"id": "n0452", "say": "सुब्रमण्यम!", "lang": "h", "file": "names/n0452.mp4"}, {"id": "n0453", "say": "सुंदर!", "lang": "h", "file": "names/n0453.mp4"}, {"id": "n0454", "say": "सूर्या!", "lang": "h", "file": "names/n0454.mp4"}, {"id": "n0455", "say": "वसंत!", "lang": "h", "file": "names/n0455.mp4"}, {"id": "n0456", "say": "वेंकट!", "lang": "h", "file": "names/n0456.mp4"}, {"id": "n0457", "say": "वेंकटेश!", "lang": "h", "file": "names/n0457.mp4"}, {"id": "n0458", "say": "विग्नेश!", "lang": "h", "file": "names/n0458.mp4"}, {"id": "n0459", "say": "विमल!", "lang": "h", "file": "names/n0459.mp4"}, {"id": "n0460", "say": "युवराज!", "lang": "h", "file": "names/n0460.mp4"}, {"id": "n0461", "say": "बाला!", "lang": "h", "file": "names/n0461.mp4"}, {"id": "n0462", "say": "चंद्रू!", "lang": "h", "file": "names/n0462.mp4"}, {"id": "n0463", "say": "कन्नन!", "lang": "h", "file": "names/n0463.mp4"}, {"id": "n0464", "say": "मणि!", "lang": "h", "file": "names/n0464.mp4"}, {"id": "n0465", "say": "रामकुमार!", "lang": "h", "file": "names/n0465.mp4"}, {"id": "n0466", "say": "राजन!", "lang": "h", "file": "names/n0466.mp4"}, {"id": "n0467", "say": "सत्या!", "lang": "h", "file": "names/n0467.mp4"}, {"id": "n0468", "say": "सुदर्शन!", "lang": "h", "file": "names/n0468.mp4"}, {"id": "n0469", "say": "सुजीत!", "lang": "h", "file": "names/n0469.mp4"}, {"id": "n0470", "say": "विनीत!", "lang": "h", "file": "names/n0470.mp4"}, {"id": "n0471", "say": "विष्णु!", "lang": "h", "file": "names/n0471.mp4"}, {"id": "n0472", "say": "बीजू!", "lang": "h", "file": "names/n0472.mp4"}, {"id": "n0473", "say": "जितिन!", "lang": "h", "file": "names/n0473.mp4"}, {"id": "n0474", "say": "श्रीजीत!", "lang": "h", "file": "names/n0474.mp4"}, {"id": "n0475", "say": "संजू!", "lang": "h", "file": "names/n0475.mp4"}, {"id": "n0476", "say": "उन्नी!", "lang": "h", "file": "names/n0476.mp4"}, {"id": "n0477", "say": "मनु!", "lang": "h", "file": "names/n0477.mp4"}, {"id": "n0478", "say": "अभिलाष!", "lang": "h", "file": "names/n0478.mp4"}, {"id": "n0479", "say": "अजमल!", "lang": "h", "file": "names/n0479.mp4"}, {"id": "n0480", "say": "फ़हद!", "lang": "h", "file": "names/n0480.mp4"}, {"id": "n0481", "say": "चैतन्य!", "lang": "h", "file": "names/n0481.mp4"}, {"id": "n0482", "say": "हर्षा!", "lang": "h", "file": "names/n0482.mp4"}, {"id": "n0483", "say": "कल्याण!", "lang": "h", "file": "names/n0483.mp4"}, {"id": "n0484", "say": "फणि!", "lang": "h", "file": "names/n0484.mp4"}, {"id": "n0485", "say": "राजू!", "lang": "h", "file": "names/n0485.mp4"}, {"id": "n0486", "say": "रामू!", "lang": "h", "file": "names/n0486.mp4"}, {"id": "n0487", "say": "साई!", "lang": "h", "file": "names/n0487.mp4"}, {"id": "n0488", "say": "तेजा!", "lang": "h", "file": "names/n0488.mp4"}, {"id": "n0489", "say": "वेणु!", "lang": "h", "file": "names/n0489.mp4"}, {"id": "n0490", "say": "वंशी!", "lang": "h", "file": "names/n0490.mp4"}, {"id": "n0491", "say": "दर्शन!", "lang": "h", "file": "names/n0491.mp4"}, {"id": "n0492", "say": "गुरु!", "lang": "h", "file": "names/n0492.mp4"}, {"id": "n0493", "say": "मंजुनाथ!", "lang": "h", "file": "names/n0493.mp4"}, {"id": "n0494", "say": "नागराज!", "lang": "h", "file": "names/n0494.mp4"}, {"id": "n0495", "say": "राघवेंद्र!", "lang": "h", "file": "names/n0495.mp4"}, {"id": "n0496", "say": "शशांक!", "lang": "h", "file": "names/n0496.mp4"}, {"id": "n0497", "say": "सुदीप!", "lang": "h", "file": "names/n0497.mp4"}, {"id": "n0498", "say": "अभिजीत!", "lang": "h", "file": "names/n0498.mp4"}, {"id": "n0499", "say": "अरिजीत!", "lang": "h", "file": "names/n0499.mp4"}, {"id": "n0500", "say": "अर्णब!", "lang": "h", "file": "names/n0500.mp4"}, {"id": "n0501", "say": "देबाशीष!", "lang": "h", "file": "names/n0501.mp4"}, {"id": "n0502", "say": "इंद्रनील!", "lang": "h", "file": "names/n0502.mp4"}, {"id": "n0503", "say": "पार्थ!", "lang": "h", "file": "names/n0503.mp4"}, {"id": "n0504", "say": "प्रीतम!", "lang": "h", "file": "names/n0504.mp4"}, {"id": "n0505", "say": "राजीब!", "lang": "h", "file": "names/n0505.mp4"}, {"id": "n0506", "say": "सायन!", "lang": "h", "file": "names/n0506.mp4"}, {"id": "n0507", "say": "सुभजीत!", "lang": "h", "file": "names/n0507.mp4"}, {"id": "n0508", "say": "तन्मय!", "lang": "h", "file": "names/n0508.mp4"}, {"id": "n0509", "say": "अनिर्बान!", "lang": "h", "file": "names/n0509.mp4"}, {"id": "n0510", "say": "बिस्वजीत!", "lang": "h", "file": "names/n0510.mp4"}, {"id": "n0511", "say": "प्रीतीश!", "lang": "h", "file": "names/n0511.mp4"}, {"id": "n0512", "say": "सुब्रत!", "lang": "h", "file": "names/n0512.mp4"}, {"id": "n0513", "say": "संजीब!", "lang": "h", "file": "names/n0513.mp4"}, {"id": "n0514", "say": "चिराग!", "lang": "h", "file": "names/n0514.mp4"}, {"id": "n0515", "say": "धवल!", "lang": "h", "file": "names/n0515.mp4"}, {"id": "n0516", "say": "हर्षद!", "lang": "h", "file": "names/n0516.mp4"}, {"id": "n0517", "say": "हार्दिक!", "lang": "h", "file": "names/n0517.mp4"}, {"id": "n0518", "say": "जिग्नेश!", "lang": "h", "file": "names/n0518.mp4"}, {"id": "n0519", "say": "कौशल!", "lang": "h", "file": "names/n0519.mp4"}, {"id": "n0520", "say": "केतन!", "lang": "h", "file": "names/n0520.mp4"}, {"id": "n0521", "say": "मेहुल!", "lang": "h", "file": "names/n0521.mp4"}, {"id": "n0522", "say": "नीरव!", "lang": "h", "file": "names/n0522.mp4"}, {"id": "n0523", "say": "प्रतीक!", "lang": "h", "file": "names/n0523.mp4"}, {"id": "n0524", "say": "ऋषिकेश!", "lang": "h", "file": "names/n0524.mp4"}, {"id": "n0525", "say": "संकेत!", "lang": "h", "file": "names/n0525.mp4"}, {"id": "n0526", "say": "सिद्धेश!", "lang": "h", "file": "names/n0526.mp4"}, {"id": "n0527", "say": "स्वप्निल!", "lang": "h", "file": "names/n0527.mp4"}, {"id": "n0528", "say": "तेजस!", "lang": "h", "file": "names/n0528.mp4"}, {"id": "n0529", "say": "ओंकार!", "lang": "h", "file": "names/n0529.mp4"}, {"id": "n0530", "say": "मिहिर!", "lang": "h", "file": "names/n0530.mp4"}, {"id": "n0531", "say": "भावेश!", "lang": "h", "file": "names/n0531.mp4"}, {"id": "n0532", "say": "कल्पेश!", "lang": "h", "file": "names/n0532.mp4"}, {"id": "n0533", "say": "परेश!", "lang": "h", "file": "names/n0533.mp4"}, {"id": "n0534", "say": "रूपेश!", "lang": "h", "file": "names/n0534.mp4"}, {"id": "n0535", "say": "मितेश!", "lang": "h", "file": "names/n0535.mp4"}, {"id": "n0536", "say": "वैभव!", "lang": "h", "file": "names/n0536.mp4"}, {"id": "n0537", "say": "अमनदीप!", "lang": "h", "file": "names/n0537.mp4"}, {"id": "n0538", "say": "अमरजीत!", "lang": "h", "file": "names/n0538.mp4"}, {"id": "n0539", "say": "अर्शदीप!", "lang": "h", "file": "names/n0539.mp4"}, {"id": "n0540", "say": "बलजीत!", "lang": "h", "file": "names/n0540.mp4"}, {"id": "n0541", "say": "गुरदीप!", "lang": "h", "file": "names/n0541.mp4"}, {"id": "n0542", "say": "हरजीत!", "lang": "h", "file": "names/n0542.mp4"}, {"id": "n0543", "say": "जसविंदर!", "lang": "h", "file": "names/n0543.mp4"}, {"id": "n0544", "say": "जगजीत!", "lang": "h", "file": "names/n0544.mp4"}, {"id": "n0545", "say": "कुलदीप!", "lang": "h", "file": "names/n0545.mp4"}, {"id": "n0546", "say": "लवप्रीत!", "lang": "h", "file": "names/n0546.mp4"}, {"id": "n0547", "say": "मनदीप!", "lang": "h", "file": "names/n0547.mp4"}, {"id": "n0548", "say": "मनजीत!", "lang": "h", "file": "names/n0548.mp4"}, {"id": "n0549", "say": "नवदीप!", "lang": "h", "file": "names/n0549.mp4"}, {"id": "n0550", "say": "परमजीत!", "lang": "h", "file": "names/n0550.mp4"}, {"id": "n0551", "say": "राजिंदर!", "lang": "h", "file": "names/n0551.mp4"}, {"id": "n0552", "say": "रणबीर!", "lang": "h", "file": "names/n0552.mp4"}, {"id": "n0553", "say": "सुखविंदर!", "lang": "h", "file": "names/n0553.mp4"}, {"id": "n0554", "say": "सुखदीप!", "lang": "h", "file": "names/n0554.mp4"}, {"id": "n0555", "say": "तेजिंदर!", "lang": "h", "file": "names/n0555.mp4"}, {"id": "n0556", "say": "इंदर!", "lang": "h", "file": "names/n0556.mp4"}, {"id": "n0557", "say": "गगन!", "lang": "h", "file": "names/n0557.mp4"}, {"id": "n0558", "say": "जसकरण!", "lang": "h", "file": "names/n0558.mp4"}, {"id": "n0559", "say": "करणवीर!", "lang": "h", "file": "names/n0559.mp4"}, {"id": "n0560", "say": "आमिर!", "lang": "h", "file": "names/n0560.mp4"}, {"id": "n0561", "say": "अब्दुल!", "lang": "h", "file": "names/n0561.mp4"}, {"id": "n0562", "say": "आदिल!", "lang": "h", "file": "names/n0562.mp4"}, {"id": "n0563", "say": "अहमद!", "lang": "h", "file": "names/n0563.mp4"}, {"id": "n0564", "say": "अकबर!", "lang": "h", "file": "names/n0564.mp4"}, {"id": "n0565", "say": "अली!", "lang": "h", "file": "names/n0565.mp4"}, {"id": "n0566", "say": "अनवर!", "lang": "h", "file": "names/n0566.mp4"}, {"id": "n0567", "say": "आरिफ़!", "lang": "h", "file": "names/n0567.mp4"}, {"id": "n0568", "say": "आसिफ़!", "lang": "h", "file": "names/n0568.mp4"}, {"id": "n0569", "say": "अयान!", "lang": "h", "file": "names/n0569.mp4"}, {"id": "n0570", "say": "फ़ैसल!", "lang": "h", "file": "names/n0570.mp4"}, {"id": "n0571", "say": "फ़रहान!", "lang": "h", "file": "names/n0571.mp4"}, {"id": "n0572", "say": "हमज़ा!", "lang": "h", "file": "names/n0572.mp4"}, {"id": "n0573", "say": "हसन!", "lang": "h", "file": "names/n0573.mp4"}, {"id": "n0574", "say": "इमरान!", "lang": "h", "file": "names/n0574.mp4"}, {"id": "n0575", "say": "इरफ़ान!", "lang": "h", "file": "names/n0575.mp4"}, {"id": "n0576", "say": "जावेद!", "lang": "h", "file": "names/n0576.mp4"}, {"id": "n0577", "say": "जुनैद!", "lang": "h", "file": "names/n0577.mp4"}, {"id": "n0578", "say": "कबीर!", "lang": "h", "file": "names/n0578.mp4"}, {"id": "n0579", "say": "ख़ालिद!", "lang": "h", "file": "names/n0579.mp4"}, {"id": "n0580", "say": "मोहम्मद!", "lang": "h", "file": "names/n0580.mp4"}, {"id": "n0581", "say": "मोहसिन!", "lang": "h", "file": "names/n0581.mp4"}, {"id": "n0582", "say": "नदीम!", "lang": "h", "file": "names/n0582.mp4"}, {"id": "n0583", "say": "नवीद!", "lang": "h", "file": "names/n0583.mp4"}, {"id": "n0584", "say": "रहीम!", "lang": "h", "file": "names/n0584.mp4"}, {"id": "n0585", "say": "राशिद!", "lang": "h", "file": "names/n0585.mp4"}, {"id": "n0586", "say": "रेहान!", "lang": "h", "file": "names/n0586.mp4"}, {"id": "n0587", "say": "रिज़वान!", "lang": "h", "file": "names/n0587.mp4"}, {"id": "n0588", "say": "साजिद!", "lang": "h", "file": "names/n0588.mp4"}, {"id": "n0589", "say": "सलमान!", "lang": "h", "file": "names/n0589.mp4"}, {"id": "n0590", "say": "शाहिद!", "lang": "h", "file": "names/n0590.mp4"}, {"id": "n0591", "say": "शाहरुख़!", "lang": "h", "file": "names/n0591.mp4"}, {"id": "n0592", "say": "सोहेल!", "lang": "h", "file": "names/n0592.mp4"}, {"id": "n0593", "say": "तारिक़!", "lang": "h", "file": "names/n0593.mp4"}, {"id": "n0594", "say": "वसीम!", "lang": "h", "file": "names/n0594.mp4"}, {"id": "n0595", "say": "यूसुफ़!", "lang": "h", "file": "names/n0595.mp4"}, {"id": "n0596", "say": "ज़ैद!", "lang": "h", "file": "names/n0596.mp4"}, {"id": "n0597", "say": "ज़ुबैर!", "lang": "h", "file": "names/n0597.mp4"}, {"id": "n0598", "say": "अरशद!", "lang": "h", "file": "names/n0598.mp4"}, {"id": "n0599", "say": "अज़हर!", "lang": "h", "file": "names/n0599.mp4"}, {"id": "n0600", "say": "फ़ैज़!", "lang": "h", "file": "names/n0600.mp4"}, {"id": "n0601", "say": "इब्राहिम!", "lang": "h", "file": "names/n0601.mp4"}, {"id": "n0602", "say": "इस्माइल!", "lang": "h", "file": "names/n0602.mp4"}, {"id": "n0603", "say": "मुस्तफ़ा!", "lang": "h", "file": "names/n0603.mp4"}, {"id": "n0604", "say": "नवाज़!", "lang": "h", "file": "names/n0604.mp4"}, {"id": "n0605", "say": "रज़ा!", "lang": "h", "file": "names/n0605.mp4"}, {"id": "n0606", "say": "शोएब!", "lang": "h", "file": "names/n0606.mp4"}, {"id": "n0607", "say": "उस्मान!", "lang": "h", "file": "names/n0607.mp4"}, {"id": "n0608", "say": "ज़ीशान!", "lang": "h", "file": "names/n0608.mp4"}, {"id": "n0609", "say": "John!", "lang": "a", "file": "names/n0609.mp4"}, {"id": "n0610", "say": "Joseph!", "lang": "a", "file": "names/n0610.mp4"}, {"id": "n0611", "say": "Thomas!", "lang": "a", "file": "names/n0611.mp4"}, {"id": "n0612", "say": "George!", "lang": "a", "file": "names/n0612.mp4"}, {"id": "n0613", "say": "Matthew!", "lang": "a", "file": "names/n0613.mp4"}, {"id": "n0614", "say": "Mathew!", "lang": "a", "file": "names/n0614.mp4"}, {"id": "n0615", "say": "Paul!", "lang": "a", "file": "names/n0615.mp4"}, {"id": "n0616", "say": "Peter!", "lang": "a", "file": "names/n0616.mp4"}, {"id": "n0617", "say": "James!", "lang": "a", "file": "names/n0617.mp4"}, {"id": "n0618", "say": "David!", "lang": "a", "file": "names/n0618.mp4"}, {"id": "n0619", "say": "Daniel!", "lang": "a", "file": "names/n0619.mp4"}, {"id": "n0620", "say": "Michael!", "lang": "a", "file": "names/n0620.mp4"}, {"id": "n0621", "say": "Mary!", "lang": "a", "file": "names/n0621.mp4"}, {"id": "n0622", "say": "Anne!", "lang": "a", "file": "names/n0622.mp4"}, {"id": "n0623", "say": "Ann!", "lang": "a", "file": "names/n0623.mp4"}, {"id": "n0624", "say": "Grace!", "lang": "a", "file": "names/n0624.mp4"}, {"id": "n0625", "say": "Rose!", "lang": "a", "file": "names/n0625.mp4"}, {"id": "n0626", "say": "Sarah!", "lang": "a", "file": "names/n0626.mp4"}, {"id": "n0627", "say": "Elizabeth!", "lang": "a", "file": "names/n0627.mp4"}, {"id": "n0628", "say": "Jessica!", "lang": "a", "file": "names/n0628.mp4"}, {"id": "n0629", "say": "Jennifer!", "lang": "a", "file": "names/n0629.mp4"}, {"id": "n0630", "say": "Emma!", "lang": "a", "file": "names/n0630.mp4"}, {"id": "n0631", "say": "Olivia!", "lang": "a", "file": "names/n0631.mp4"}, {"id": "n0632", "say": "Sophia!", "lang": "a", "file": "names/n0632.mp4"}, {"id": "n0633", "say": "Emily!", "lang": "a", "file": "names/n0633.mp4"}, {"id": "n0634", "say": "Anna!", "lang": "a", "file": "names/n0634.mp4"}, {"id": "n0635", "say": "Maria!", "lang": "a", "file": "names/n0635.mp4"}, {"id": "n0636", "say": "Linda!", "lang": "a", "file": "names/n0636.mp4"}, {"id": "n0637", "say": "Susan!", "lang": "a", "file": "names/n0637.mp4"}, {"id": "n0638", "say": "Rachel!", "lang": "a", "file": "names/n0638.mp4"}, {"id": "n0639", "say": "Rebecca!", "lang": "a", "file": "names/n0639.mp4"}, {"id": "n0640", "say": "Christina!", "lang": "a", "file": "names/n0640.mp4"}, {"id": "n0641", "say": "Christopher!", "lang": "a", "file": "names/n0641.mp4"}, {"id": "n0642", "say": "Robert!", "lang": "a", "file": "names/n0642.mp4"}, {"id": "n0643", "say": "William!", "lang": "a", "file": "names/n0643.mp4"}, {"id": "n0644", "say": "Richard!", "lang": "a", "file": "names/n0644.mp4"}, {"id": "n0645", "say": "Charles!", "lang": "a", "file": "names/n0645.mp4"}, {"id": "n0646", "say": "Mark!", "lang": "a", "file": "names/n0646.mp4"}, {"id": "n0647", "say": "Steven!", "lang": "a", "file": "names/n0647.mp4"}, {"id": "n0648", "say": "Stephen!", "lang": "a", "file": "names/n0648.mp4"}, {"id": "n0649", "say": "Andrew!", "lang": "a", "file": "names/n0649.mp4"}, {"id": "n0650", "say": "Anthony!", "lang": "a", "file": "names/n0650.mp4"}, {"id": "n0651", "say": "Kevin!", "lang": "a", "file": "names/n0651.mp4"}, {"id": "n0652", "say": "Brian!", "lang": "a", "file": "names/n0652.mp4"}, {"id": "n0653", "say": "Jason!", "lang": "a", "file": "names/n0653.mp4"}, {"id": "n0654", "say": "Ryan!", "lang": "a", "file": "names/n0654.mp4"}, {"id": "n0655", "say": "Eric!", "lang": "a", "file": "names/n0655.mp4"}, {"id": "n0656", "say": "Alex!", "lang": "a", "file": "names/n0656.mp4"}, {"id": "n0657", "say": "Alexander!", "lang": "a", "file": "names/n0657.mp4"}, {"id": "n0658", "say": "Sam!", "lang": "a", "file": "names/n0658.mp4"}, {"id": "n0659", "say": "Samuel!", "lang": "a", "file": "names/n0659.mp4"}, {"id": "n0660", "say": "Benjamin!", "lang": "a", "file": "names/n0660.mp4"}, {"id": "n0661", "say": "Lucas!", "lang": "a", "file": "names/n0661.mp4"}, {"id": "n0662", "say": "Liam!", "lang": "a", "file": "names/n0662.mp4"}, {"id": "n0663", "say": "Noah!", "lang": "a", "file": "names/n0663.mp4"}, {"id": "n0664", "say": "Oliver!", "lang": "a", "file": "names/n0664.mp4"}, {"id": "n0665", "say": "Ethan!", "lang": "a", "file": "names/n0665.mp4"}, {"id": "n0666", "say": "Jacob!", "lang": "a", "file": "names/n0666.mp4"}, {"id": "n0667", "say": "Ava!", "lang": "a", "file": "names/n0667.mp4"}, {"id": "n0668", "say": "Mia!", "lang": "a", "file": "names/n0668.mp4"}, {"id": "n0669", "say": "Isabella!", "lang": "a", "file": "names/n0669.mp4"}, {"id": "n0670", "say": "Charlotte!", "lang": "a", "file": "names/n0670.mp4"}, {"id": "n0671", "say": "Amelia!", "lang": "a", "file": "names/n0671.mp4"}, {"id": "n0672", "say": "Lily!", "lang": "a", "file": "names/n0672.mp4"}, {"id": "n0673", "say": "Chloe!", "lang": "a", "file": "names/n0673.mp4"}, {"id": "n0674", "say": "Ella!", "lang": "a", "file": "names/n0674.mp4"}, {"id": "n0675", "say": "Zoe!", "lang": "a", "file": "names/n0675.mp4"}, {"id": "n0676", "say": "Hannah!", "lang": "a", "file": "names/n0676.mp4"}, {"id": "n0677", "say": "Laura!", "lang": "a", "file": "names/n0677.mp4"}, {"id": "n0678", "say": "Julia!", "lang": "a", "file": "names/n0678.mp4"}, {"id": "n0679", "say": "Natalie!", "lang": "a", "file": "names/n0679.mp4"}, {"id": "n0680", "say": "Victoria!", "lang": "a", "file": "names/n0680.mp4"}, {"id": "n0681", "say": "Nancy!", "lang": "a", "file": "names/n0681.mp4"}, {"id": "n0682", "say": "Karen!", "lang": "a", "file": "names/n0682.mp4"}, {"id": "n0683", "say": "Lisa!", "lang": "a", "file": "names/n0683.mp4"}, {"id": "n0684", "say": "Michelle!", "lang": "a", "file": "names/n0684.mp4"}, {"id": "n0685", "say": "Angela!", "lang": "a", "file": "names/n0685.mp4"}, {"id": "n0686", "say": "Helen!", "lang": "a", "file": "names/n0686.mp4"}, {"id": "n0687", "say": "Catherine!", "lang": "a", "file": "names/n0687.mp4"}, {"id": "n0688", "say": "Kate!", "lang": "a", "file": "names/n0688.mp4"}, {"id": "n0689", "say": "Katie!", "lang": "a", "file": "names/n0689.mp4"}, {"id": "n0690", "say": "Amy!", "lang": "a", "file": "names/n0690.mp4"}, {"id": "n0691", "say": "Claire!", "lang": "a", "file": "names/n0691.mp4"}, {"id": "n0692", "say": "Lucy!", "lang": "a", "file": "names/n0692.mp4"}, {"id": "n0693", "say": "Alice!", "lang": "a", "file": "names/n0693.mp4"}, {"id": "n0694", "say": "Diana!", "lang": "a", "file": "names/n0694.mp4"}, {"id": "n0695", "say": "Nicole!", "lang": "a", "file": "names/n0695.mp4"}, {"id": "n0696", "say": "Stephanie!", "lang": "a", "file": "names/n0696.mp4"}, {"id": "n0697", "say": "Tom!", "lang": "a", "file": "names/n0697.mp4"}, {"id": "n0698", "say": "Tony!", "lang": "a", "file": "names/n0698.mp4"}, {"id": "n0699", "say": "Jack!", "lang": "a", "file": "names/n0699.mp4"}, {"id": "n0700", "say": "Harry!", "lang": "a", "file": "names/n0700.mp4"}, {"id": "n0701", "say": "Henry!", "lang": "a", "file": "names/n0701.mp4"}, {"id": "n0702", "say": "Edward!", "lang": "a", "file": "names/n0702.mp4"}, {"id": "n0703", "say": "Frank!", "lang": "a", "file": "names/n0703.mp4"}, {"id": "n0704", "say": "Simon!", "lang": "a", "file": "names/n0704.mp4"}, {"id": "n0705", "say": "Martin!", "lang": "a", "file": "names/n0705.mp4"}, {"id": "n0706", "say": "Patrick!", "lang": "a", "file": "names/n0706.mp4"}, {"id": "n0707", "say": "Sean!", "lang": "a", "file": "names/n0707.mp4"}, {"id": "n0708", "say": "Adam!", "lang": "a", "file": "names/n0708.mp4"}, {"id": "n0709", "say": "Aaron!", "lang": "a", "file": "names/n0709.mp4"}, {"id": "n0710", "say": "Nathan!", "lang": "a", "file": "names/n0710.mp4"}, {"id": "n0711", "say": "Justin!", "lang": "a", "file": "names/n0711.mp4"}, {"id": "n0712", "say": "Jonathan!", "lang": "a", "file": "names/n0712.mp4"}, {"id": "n0713", "say": "Joshua!", "lang": "a", "file": "names/n0713.mp4"}, {"id": "n0714", "say": "Luke!", "lang": "a", "file": "names/n0714.mp4"}, {"id": "n0715", "say": "Philip!", "lang": "a", "file": "names/n0715.mp4"}, {"id": "n0716", "say": "Vincent!", "lang": "a", "file": "names/n0716.mp4"}, {"id": "n0717", "say": "Victor!", "lang": "a", "file": "names/n0717.mp4"}, {"id": "n0718", "say": "Felix!", "lang": "a", "file": "names/n0718.mp4"}, {"id": "n0719", "say": "Max!", "lang": "a", "file": "names/n0719.mp4"}, {"id": "n0720", "say": "Leo!", "lang": "a", "file": "names/n0720.mp4"}, {"id": "n0721", "say": "Oscar!", "lang": "a", "file": "names/n0721.mp4"}, {"id": "n0722", "say": "Carlos!", "lang": "a", "file": "names/n0722.mp4"}, {"id": "n0723", "say": "Juan!", "lang": "a", "file": "names/n0723.mp4"}, {"id": "n0724", "say": "Jose!", "lang": "a", "file": "names/n0724.mp4"}, {"id": "n0725", "say": "Luis!", "lang": "a", "file": "names/n0725.mp4"}, {"id": "n0726", "say": "Miguel!", "lang": "a", "file": "names/n0726.mp4"}, {"id": "n0727", "say": "Pedro!", "lang": "a", "file": "names/n0727.mp4"}, {"id": "n0728", "say": "Diego!", "lang": "a", "file": "names/n0728.mp4"}, {"id": "n0729", "say": "Omar!", "lang": "a", "file": "names/n0729.mp4"}, {"id": "n0730", "say": "Yuki!", "lang": "a", "file": "names/n0730.mp4"}, {"id": "n0731", "say": "Hiroshi!", "lang": "a", "file": "names/n0731.mp4"}, {"id": "n0732", "say": "Kenji!", "lang": "a", "file": "names/n0732.mp4"}, {"id": "n0733", "say": "Wei!", "lang": "a", "file": "names/n0733.mp4"}, {"id": "n0734", "say": "Mei!", "lang": "a", "file": "names/n0734.mp4"}, {"id": "n0735", "say": "Hans!", "lang": "a", "file": "names/n0735.mp4"}, {"id": "n0736", "say": "Pierre!", "lang": "a", "file": "names/n0736.mp4"}, {"id": "n0737", "say": "Jean!", "lang": "a", "file": "names/n0737.mp4"}, {"id": "n0738", "say": "Marie!", "lang": "a", "file": "names/n0738.mp4"}, {"id": "n0739", "say": "Sophie!", "lang": "a", "file": "names/n0739.mp4"}, {"id": "n0740", "say": "Giulia!", "lang": "a", "file": "names/n0740.mp4"}, {"id": "n0741", "say": "Marco!", "lang": "a", "file": "names/n0741.mp4"}, {"id": "n0742", "say": "Luca!", "lang": "a", "file": "names/n0742.mp4"}, {"id": "n0743", "say": "Paolo!", "lang": "a", "file": "names/n0743.mp4"}, {"id": "n0744", "say": "Francesco!", "lang": "a", "file": "names/n0744.mp4"}, {"id": "n0745", "say": "Elena!", "lang": "a", "file": "names/n0745.mp4"}, {"id": "n0746", "say": "Ana!", "lang": "a", "file": "names/n0746.mp4"}, {"id": "n0747", "say": "Lucia!", "lang": "a", "file": "names/n0747.mp4"}, {"id": "n0748", "say": "Isabel!", "lang": "a", "file": "names/n0748.mp4"}, {"id": "n0749", "say": "Kim!", "lang": "a", "file": "names/n0749.mp4"}, {"id": "n0750", "say": "Steve!", "lang": "a", "file": "names/n0750.mp4"}, {"id": "n0751", "say": "Joe!", "lang": "a", "file": "names/n0751.mp4"}, {"id": "n0752", "say": "Ben!", "lang": "a", "file": "names/n0752.mp4"}, {"id": "n0753", "say": "Dan!", "lang": "a", "file": "names/n0753.mp4"}, {"id": "n0754", "say": "Chris!", "lang": "a", "file": "names/n0754.mp4"}, {"id": "n0755", "say": "Matt!", "lang": "a", "file": "names/n0755.mp4"}, {"id": "n0756", "say": "Mike!", "lang": "a", "file": "names/n0756.mp4"}, {"id": "n0757", "say": "Nick!", "lang": "a", "file": "names/n0757.mp4"}, {"id": "n0758", "say": "Rob!", "lang": "a", "file": "names/n0758.mp4"}, {"id": "n0759", "say": "Tim!", "lang": "a", "file": "names/n0759.mp4"}, {"id": "n0760", "say": "Will!", "lang": "a", "file": "names/n0760.mp4"}, {"id": "n0761", "say": "Jim!", "lang": "a", "file": "names/n0761.mp4"}, {"id": "n0762", "say": "Bob!", "lang": "a", "file": "names/n0762.mp4"}, {"id": "n0763", "say": "Jenny!", "lang": "a", "file": "names/n0763.mp4"}, {"id": "n0764", "say": "Annie!", "lang": "a", "file": "names/n0764.mp4"}, {"id": "n0765", "say": "Nina!", "lang": "a", "file": "names/n0765.mp4"}, {"id": "n0766", "say": "Rita!", "lang": "a", "file": "names/n0766.mp4"}, {"id": "n0767", "say": "Ruby!", "lang": "a", "file": "names/n0767.mp4"}, {"id": "n0768", "say": "Ivy!", "lang": "a", "file": "names/n0768.mp4"}, {"id": "n0769", "say": "Jasmine!", "lang": "a", "file": "names/n0769.mp4"}, {"id": "n0770", "say": "Joyce!", "lang": "a", "file": "names/n0770.mp4"}, {"id": "n0771", "say": "Gloria!", "lang": "a", "file": "names/n0771.mp4"}, {"id": "n0772", "say": "Teresa!", "lang": "a", "file": "names/n0772.mp4"}, {"id": "n0773", "say": "Theresa!", "lang": "a", "file": "names/n0773.mp4"}, {"id": "n0774", "say": "Agnes!", "lang": "a", "file": "names/n0774.mp4"}, {"id": "n0775", "say": "Shirley!", "lang": "a", "file": "names/n0775.mp4"}, {"id": "n0776", "say": "Sharon!", "lang": "a", "file": "names/n0776.mp4"}, {"id": "n0777", "say": "Jerry!", "lang": "a", "file": "names/n0777.mp4"}, {"id": "n0778", "say": "Jeffrey!", "lang": "a", "file": "names/n0778.mp4"}, {"id": "n0779", "say": "Larry!", "lang": "a", "file": "names/n0779.mp4"}, {"id": "n0780", "say": "Gary!", "lang": "a", "file": "names/n0780.mp4"}, {"id": "n0781", "say": "Dennis!", "lang": "a", "file": "names/n0781.mp4"}, {"id": "n0782", "say": "Walter!", "lang": "a", "file": "names/n0782.mp4"}, {"id": "n0783", "say": "Albert!", "lang": "a", "file": "names/n0783.mp4"}, {"id": "n0784", "say": "Arthur!", "lang": "a", "file": "names/n0784.mp4"}, {"id": "n0785", "say": "Lawrence!", "lang": "a", "file": "names/n0785.mp4"}, {"id": "n0786", "say": "Roger!", "lang": "a", "file": "names/n0786.mp4"}, {"id": "n0787", "say": "Carl!", "lang": "a", "file": "names/n0787.mp4"}, {"id": "n0788", "say": "Ray!", "lang": "a", "file": "names/n0788.mp4"}, {"id": "n0789", "say": "Wayne!", "lang": "a", "file": "names/n0789.mp4"}, {"id": "n0790", "say": "Alan!", "lang": "a", "file": "names/n0790.mp4"}, {"id": "n0791", "say": "Ralph!", "lang": "a", "file": "names/n0791.mp4"}, {"id": "n0792", "say": "Roy!", "lang": "a", "file": "names/n0792.mp4"}, {"id": "n0793", "say": "Louis!", "lang": "a", "file": "names/n0793.mp4"}, {"id": "n0794", "say": "Eugene!", "lang": "a", "file": "names/n0794.mp4"}, {"id": "n0795", "say": "Russell!", "lang": "a", "file": "names/n0795.mp4"}, {"id": "n0796", "say": "Bruce!", "lang": "a", "file": "names/n0796.mp4"}]''')
json.dump(JOBS, open('/content/jobs.json', 'w'), ensure_ascii=False)
print(len(JOBS), 'clips to make,', sum(os.path.exists(W + '/web/' + j['file']) for j in JOBS), 'already done')


In [ ]:
%%writefile /content/voices.py
# 5. Her voice for each clip (Kokoro, af_bella; names with the Indian pronunciation)
import json, os, numpy as np, soundfile as sf
from kokoro import KPipeline
W = '/content/drive/MyDrive/hannah_clips'
jobs = json.load(open('/content/jobs.json'))
pipes = {}
for i, j in enumerate(jobs):
    out = '%s/wav/%s.wav' % (W, j['id'])
    if os.path.exists(out) or os.path.exists(W + '/web/' + j['file']): continue
    try:
        if j['lang'] not in pipes: pipes[j['lang']] = KPipeline(lang_code=j['lang'])
        a = np.concatenate([np.asarray(x) for _, _, x in pipes[j['lang']](j['say'], voice='af_bella', speed=0.95)])
        pad = np.zeros(int(24000 * (0.35 if j['id'].startswith('c') else 0.3)), dtype=np.float32)
        sf.write(out, np.concatenate([pad, a, pad]), 24000)
    except Exception as e:
        print('VOICE FAILED', j['id'], repr(e)[:150], flush=True)
    if i % 50 == 0: print('voices %d/%d' % (i + 1, len(jobs)), flush=True)
print('VOICES OK')


In [ ]:
!/content/venv/bin/python /content/voices.py 2>&1 | grep -E "voices|OK|FAILED|Error"


In [ ]:
%%writefile /content/videos.py
# 6. A talking clip for each one (Ditto), easing in and out of her resting pose
import os, sys, json, copy, subprocess, traceback
os.chdir('/content/ditto'); sys.path.insert(0, '/content/ditto')
from stream_pipeline_offline import StreamSDK
from inference import run
CFG, ROOT = './checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl', './checkpoints/ditto_pytorch'
MORE = {'setup_kwargs': {'fade_type': 's', 'fade_out_keys': ('exp', 'pitch', 'yaw', 'roll', 't')},
        'run_kwargs': {'fade_in': 8, 'fade_out': 8}}
W = '/content/drive/MyDrive/hannah_clips'
jobs = json.load(open('/content/jobs.json'))
os.makedirs('/content/raw', exist_ok=True)
sdk = StreamSDK(CFG, ROOT)
failed = []
for i, j in enumerate(jobs):
    web, wav, raw = W + '/web/' + j['file'], '%s/wav/%s.wav' % (W, j['id']), '/content/raw/%s.mp4' % j['id']
    if os.path.exists(web): continue
    if not os.path.exists(wav): failed.append(j['id']); continue
    try:
        try:
            run(sdk, wav, '/content/hannah.jpg', raw, more_kwargs=copy.deepcopy(MORE))
        except Exception:
            traceback.print_exc(); sdk = StreamSDK(CFG, ROOT)
            run(sdk, wav, '/content/hannah.jpg', raw, more_kwargs=copy.deepcopy(MORE))
        tmp = '/content/raw/%s.web.mp4' % j['id']
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', raw, '-vf', 'scale=1280:-2', '-c:v', 'libx264',
                        '-preset', 'slow', '-crf', '23', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '96k',
                        '-movflags', '+faststart', tmp], check=True)
        subprocess.run(['cp', tmp, web], check=True)          # into Drive only when complete
        print('%d/%d %s ok' % (i + 1, len(jobs), j['id']), flush=True)
    except Exception:
        traceback.print_exc(); failed.append(j['id'])
print('FAILED:', failed if failed else 'none')
print('VIDEOS OK' if not failed else 'SOME VIDEOS FAILED: send Claude a screenshot')


In [ ]:
!/content/venv/bin/python /content/videos.py 2>&1 | grep -E "ok$|OK|FAILED|Error|error"


In [ ]:
# 8. Download what is ready (you can run this any time)
import json, os, subprocess
from google.colab import files
jobs = json.load(open('/content/jobs.json'))
done = [j for j in jobs if os.path.exists(W + '/web/' + j['file'])]
s = [j['file'] for j in done if j['id'].startswith('c')]
n = [j['file'] for j in done if j['id'].startswith('n')]
print(len(s), 'sentences and', len(n), 'names ready, of', len(jobs))
for zname, lst in [('hannah_short_clips.zip', s), ('hannah_names.zip', n)]:
    if not lst: continue
    if os.path.exists('/content/' + zname): os.remove('/content/' + zname)
    subprocess.run(['zip', '-q', '/content/' + zname] + lst, cwd=W + '/web', check=True)
    files.download('/content/' + zname)
